# 4. Архитектура нейросети для генерации юмористических текстов

В этом ноутбуке проектируется архитектура нейросети, которая принимает на вход семантический граф (построенный из македонских текстов) и генерирует юмористические тексты на македонском языке. Теоретическая база опирается на две книги: Bellegarda "Latent Semantic Mapping" (LSM) и Barrat, Barthelemy, Vespignani "Dynamical Processes on Complex Networks".

Структура ноутбука:
- Теоретическая часть: конспекты книг и связь теории с проектом (Фаза A)
- Сравнение вариантов архитектуры и выбор основного подхода (Фаза B)
- Диаграмма потоков данных и подготовка данных (Фаза C)
- Прототип нейросети и обучение (Фаза D)
- Baseline-модели и сравнение результатов (Фаза E)
- Ограничения и выводы (Фаза F)

## Часть 1. Теоретическая база: Latent Semantic Mapping

### 1.1. Конспект книги "Latent Semantic Mapping: Principles & Applications" (главы 1-5)

Автор: Jerome R. Bellegarda (Apple Inc.), 2007 год.

Книга про то, как автоматически находить скрытые смысловые связи между словами и документами, используя математику линейной алгебры. Ниже конспект первых пяти глав (Part I: Principles).

#### Глава 1. Введение: зачем нужна LSA и как из нее выросла LSM

Проблема простая: когда мы ищем документы по ключевым словам, мы часто не находим нужное. Почему? Потому что одну и ту же мысль можно выразить разными словами (это снижает recall, полноту поиска). А одно слово может означать разные вещи (это снижает precision, точность поиска).

**LSA (Latent Semantic Analysis)** решает эту проблему. Идея: за случайным выбором слов в текстах скрыта какая-то глубинная смысловая структура. Если мы ее найдем и уберем "шум" (случайность выбора слов), то получим компактное описание слов и документов, которое отражает смысл, а не поверхностные совпадения.

Как это помогает? После LSA слова и документы, которые близки по смыслу, оказываются рядом в новом пространстве, даже если у них нет общих слов. Например, документ про "автомобиль" окажется рядом с запросом про "машину".

**LSM (Latent Semantic Mapping)** это обобщение LSA. Разница такая:
- LSA работает конкретно со словами и текстовыми документами
- LSM работает с любыми "единицами" (units) и "композициями" (compositions), не обязательно текстовыми

Три ключевых свойства LSM, которые делают подход полезным:
1. Дискретные сущности (слова, документы) отображаются в непрерывное векторное пространство, где можно применять алгоритмы машинного обучения
2. Отображение определяется глобальными корреляционными паттернами (то есть учитывается весь корпус целиком, а не отдельные документы)
3. Снижение размерности встроено в сам процесс

#### Глава 2. Как устроена LSM: матрица, SVD, интерпретация

**Шаг 1: Строим матрицу совместной встречаемости W**

У нас есть M единиц (units, в нашем случае слова/леммы) и N композиций (compositions, в нашем случае документы/тексты). Строим матрицу W размером M x N, где каждая ячейка w(i,j) показывает, насколько слово i характерно для документа j.

В книге ячейка w(i,j) вычисляется как:

```
w(i,j) = (1 - epsilon_i) * kappa(i,j) / lambda_j
```

Где:
- `kappa(i,j)` это сколько раз слово i встретилось в документе j
- `lambda_j` это общее число слов в документе j (нормализация по длине документа)
- `epsilon_i` это нормализованная энтропия слова i по всей коллекции (значение от 0 до 1)

Зачем нужна энтропия `epsilon_i`? Она работает как глобальный вес: если слово встречается равномерно по всем документам (epsilon близко к 1), то вес `1 - epsilon` будет маленьким (это слово неинформативно, как стоп-слово "и", "в", "на"). А если слово встречается только в нескольких документах (epsilon близко к 0), то его вес будет большим (это характерное, информативное слово).

По сути, эта формула делает то же, что TF-IDF или BM25, только немного по-другому. В нашем проекте мы будем использовать BM25-скоры вместо формулы из книги, но принцип тот же: матрица W кодирует "насколько каждое слово важно для каждого документа".

**Шаг 2: SVD-разложение матрицы W**

Теперь ключевой шаг. Берем матрицу W размером M x N и раскладываем ее на три множителя:

```
W ≈ U * S * V^T
```

Где:
- **U** это матрица размером M x R (M слов, R измерений). Каждая строка это координаты слова в новом пространстве
- **S** (Sigma) это диагональная матрица размером R x R. На диагонали стоят сингулярные значения s1 >= s2 >= ... >= sR > 0, упорядоченные по убыванию
- **V^T** это матрица размером R x N (R измерений, N документов). Каждый столбец это координаты документа в новом пространстве
- **R** это ранг разложения, выбираемый нами. R < min(M, N)

Важно: это приближенное разложение (знак "приблизительно равно"). Мы намеренно берем R гораздо меньше, чем полный ранг матрицы W. Это и есть снижение размерности.

**Что означает каждая компонента**

**U (левая сингулярная матрица, M x R):** Каждая строка u_i это координаты слова r_i в R-мерном пространстве. Но это еще не финальные эмбеддинги слов. Финальный вектор слова: `u_i * S` (строка U, умноженная на Sigma).

**V (правая сингулярная матрица, N x R):** Каждая строка v_j это координаты документа c_j в R-мерном пространстве. Финальный вектор документа: `v_j * S` (строка V, умноженная на Sigma).

**S (матрица сингулярных значений, R x R):** Диагональная матрица. Сингулярные значения s_k показывают "важность" каждого измерения. Первое измерение (s_1) захватывает больше всего вариации в данных, второе (s_2) следующую по величине, и так далее. Каждое последующее измерение вносит все меньший вклад.

Физический смысл: можно записать разложение как сумму:

```
W ≈ s_1 * Theta_1 * Xi_1^T + s_2 * Theta_2 * Xi_2^T + ... + s_R * Theta_R * Xi_R^T
```

Каждое слагаемое (triplet) это "слой" информации. Первое слагаемое с s_1 захватывает самый сильный паттерн в данных, второе следующий по силе, и так далее. Убирая последние (слабые) слагаемые, мы убираем "шум" и оставляем "сигнал".

**Почему слова и документы оказываются в одном пространстве**

Это самое красивое свойство SVD в контексте LSM. Строки `u_i * S` (эмбеддинги слов) и строки `v_j * S` (эмбеддинги документов) живут в одном и том же R-мерном пространстве L. Это значит, что можно напрямую сравнивать слова с документами, слова со словами, документы с документами, используя одну и ту же метрику.

**Как выбирать ранг R (он же k)**

Книга говорит: R надо подбирать эмпирически. Есть верхняя и нижняя границы:
- Сверху: R ограничен рангом матрицы W (и значением min(M, N))
- Снизу: R должен быть достаточно большим, чтобы не потерять слишком много информации

Типичные значения R из книги: от 100 до 1000. Для разных задач оптимум разный.

Есть компромисс: маленький R лучше убирает шум, но теряет полезную информацию. Большой R сохраняет больше информации, но оставляет больше шума. Нужно найти баланс.

Один из способов выбрать R: посмотреть на сингулярные значения. Если после какого-то номера k значения s_k резко падают ("колено" на графике), то это хороший кандидат для R.

#### Глава 3. Пространство признаков LSM: метрики и расширение

**Как измерять близость**

Раз слова и документы теперь векторы, нужны метрики для сравнения.

**Близость двух слов** r_i и r_j: берем косинусное сходство их эмбеддингов u_i\*S и u_j\*S. Если K(r_i, r_j) = 1, эти слова всегда встречаются в одинаковых контекстах. Чем значение меньше, тем слова употребляются в более разных контекстах.

**Близость двух документов** c_i и c_j: аналогично, косинусное сходство v_i\*S и v_j\*S.

**Близость слова и документа** r_i и c_j: тоже косинусное сходство, но с поправкой. Используются вектора u_i\*S^(1/2) и v_j\*S^(1/2), то есть масштабирование корнем из Sigma вместо полной Sigma.

**Как работать с новыми документами**

Что делать, если пришел новый документ, которого не было при обучении? Не нужно пересчитывать SVD. Достаточно:

1. Построить для нового документа вектор весов по всем словам из словаря (как столбец матрицы W)
2. Умножить его на U (транспонированную левую сингулярную матрицу), чтобы получить координаты в пространстве L

Формула: `v_new = c_new^T * U`. Такой документ называется "псевдокомпозицией" (pseudo-composition).

#### Глава 4. Вычислительные затраты

Матрица W в типичных NLP-задачах разреженная (sparse): заполнено только 0.25-0.5% ячеек. Для разреженных матриц используют итеративные алгоритмы (Ланцош, подпространственные итерации), которые гораздо быстрее классического SVD. Для типичных значений M и N (тысячи-десятки тысяч) SVD считается за минуты.

Альтернативы:
- **Инкрементальный SVD:** "дообучение" по новым документам без полного пересчета
- **Random Indexing:** случайные почти ортогональные направления вместо точных главных компонент
- **Спектральная теория графов:** использование лапласиана для нелинейного снижения размерности

#### Глава 5. Вероятностные расширения

SVD-разложение имеет вероятностную интерпретацию: если предположить, что документы распределены по гауссовой смеси, то максимизация правдоподобия приводит к тем же сингулярным векторам.

**PLSA (Probabilistic Latent Semantic Analysis)** решает проблему отрицательных чисел в SVD через неотрицательную факторизацию. Обучение через EM-алгоритм. Ограничения: не работает с новыми документами, переобучение, локальные минимумы.

Более продвинутая альтернатива: **LDA (Latent Dirichlet Allocation)** с байесовским подходом.

#### Итого: ключевые идеи из Part I

1. LSM/LSA это способ найти скрытую смысловую структуру в данных
2. Строим матрицу "слова x документы" с весами (TF-IDF, BM25, или формула из книги)
3. Применяем SVD и берем только R главных компонент
4. Получаем эмбеддинги слов (строки U\*S) и документов (строки V\*S) в одном R-мерном пространстве
5. Близкие по смыслу слова/документы оказываются рядом в этом пространстве
6. Новые документы можно проецировать в то же пространство без пересчета SVD
7. R (ранг) выбирается эмпирически, типичные значения 100-1000

### 1.2. Связь LSM/SVD с нашим проектом

Здесь объясняется, как теория из книги Bellegarda применяется конкретно к нашему проекту: генерация юмористических текстов на македонском языке.

#### Наши "документы" и "слова"

В терминологии книги:
- **Compositions (документы)** = наши 98 отфильтрованных македонских художественных текстов
- **Units (единицы)** = леммы (базовые формы слов), полученные через CLASSLA

Когда CLASSLA обрабатывает текст, она разбивает его на токены и приводит каждый к лемме. Например, из слова "книгите" (книги, с определенным артиклем) получается лемма "книга". В итоге каждый текст превращается в набор лемм.

#### Как строится матрица W

Матрица W это таблица, где строки это документы (98 штук), а столбцы это уникальные леммы (примерно 10 000-15 000 штук после фильтрации стоп-слов и редких слов).

В книге Bellegarda для заполнения матрицы используется формула с энтропией. В ноутбуке `Preprocessing_SVD_example.ipynb` для этого используется TF-IDF (через `TfidfVectorizer` из sklearn). В нашем проекте мы заменяем TF-IDF на **BM25-скоры**.

Почему BM25, а не TF-IDF? BM25 лучше работает с текстами разной длины и с короткими текстами. У BM25 есть встроенная нормализация по длине документа (параметр b) и насыщение частоты (параметр k1): после определенного числа повторений слова его вес перестает расти. TF-IDF такого не умеет.

Но принцип один и тот же: в ячейке W[i][j] стоит число, которое показывает, насколько лемма j важна для документа i. Большое число значит "эта лемма характерна для этого документа". Маленькое или ноль значит "эта лемма тут не встречается или неважна".

Матрица W получается разреженной (sparse): в большинстве ячеек стоят нули, потому что конкретная лемма встречается только в нескольких документах из 98.

#### Что дает SVD-разложение

Берем матрицу W (98 документов x ~12 000 лемм) и раскладываем:

```
W ≈ U * Sigma * V^T
```

Получаем три матрицы. Вот что каждая из них значит для нашего проекта:

**U (98 x k) это координаты документов.** Каждая строка матрицы U описывает один документ из наших 98 текстов. Вместо ~12 000 чисел (по одному на каждую лемму) каждый документ теперь описывается всего k числами (например, k = 50). Финальный эмбеддинг документа: строка U, умноженная на Sigma.

**V^T (k x ~12 000) это координаты слов.** Каждый столбец V^T (или, что то же самое, каждая строка V) описывает одну лемму. Финальный эмбеддинг слова: строка V, умноженная на Sigma. На практике в ноутбуке SVD-словарь строится как `(Sigma * V^T)^T`, то есть каждая строка результата это эмбеддинг одного слова размерности k.

**Sigma (k x k) это важность каждого измерения.** Диагональная матрица с k числами на диагонали, упорядоченными по убыванию. Можно думать так: первое измерение это самое важное "направление" в смысловом пространстве наших текстов. Может быть, оно различает прозу и поэзию. Второе следующее по важности. И так далее.

#### Зачем это для LSTM-нейросети

Наша цель: обучить LSTM-нейросеть генерировать текст на македонском языке. LSTM нужны числовые представления слов (эмбеддинги) на входе.

SVD-эмбеддинги слов (из V^T * Sigma) это как раз такие представления. Каждое слово из корпуса получает вектор длины k, который отражает его смысловые связи с другими словами и документами.

Почему не использовать FastText для LSTM? FastText мы используем для графового пространства (семантический граф). А для LSTM нужны эмбеддинги, построенные именно на нашем корпусе, потому что:
- SVD-эмбеддинги отражают специфику наших 98 текстов (художественная проза на македонском)
- Можно менять размерность k без переобучения (уникальное свойство SVD)
- SVD-эмбеддинги дополняют FastText: FastText знает про морфологию (subword), SVD знает про контекст в нашем корпусе

#### Весь pipeline в одной картинке

```
98 текстов -> CLASSLA (лемматизация) -> набор лемм для каждого текста
                                              |
                                              v
                              BM25-матрица W (98 x ~12 000)
                                              |
                                              v
                                    SVD-разложение (ранг k)
                                     /        |        \
                                    /         |         \
                                   v          v          v
                              U (98 x k)   Sigma (k)   V^T (k x ~12 000)
                                   |                         |
                                   v                         v
                          эмбеддинги            эмбеддинги слов
                          документов            (словарь: лемма -> вектор)
                                                        |
                                                        v
                                              входные данные для LSTM
```

### 1.3. Привязка теории LSM к ноутбуку Preprocessing_SVD_example.ipynb

Здесь объясняется, как конкретные функции из ноутбука соотносятся с теорией из книги Bellegarda, и что мы будем менять при адаптации для нашего проекта.

#### Функция make_matrix_W_list_of_words()

**Что делает в ноутбуке:** Строит матрицу W (term-document matrix) из текстового корпуса. Внутри использует `TfidfVectorizer` из sklearn. Принимает параметры `min_df` (минимальная частота слова), `max_df` (максимальная частота), `token_pattern` (какие символы считать частью слова). Возвращает два объекта: разреженную матрицу W и список слов `words_list`.

**Что соответствует в книге:** Построение матрицы совместной встречаемости W из главы 2 (раздел 2.1). В книге ячейка w(i,j) вычисляется через формулу с энтропией. В ноутбуке вместо этой формулы используется TF-IDF.

**Что мы меняем:** Заменяем `TfidfVectorizer` на BM25-скоры. Параметры BM25:
- `k1` (обычно 1.2-2.0) контролирует насыщение частоты термина. Аналог `sublinear_tf=True` в TF-IDF
- `b` (обычно 0.75) контролирует нормализацию по длине документа
- `min_df` и `max_df` остаются для фильтрации по частоте слов

Также подаем уже лемматизированные тексты (токенизацию делает CLASSLA, а не vectorizer).

#### Функция apply_svd()

**Что делает в ноутбуке:** Принимает матрицу W и ранг k. Вызывает `svds(W, k)` из `scipy.sparse.linalg`. Получает три компоненты: u, sigma, vt. Потом сортирует сингулярные значения по убыванию (потому что `svds` не гарантирует порядок). Возвращает матрицу `(Sigma * V^T)^T` это и есть SVD-эмбеддинги слов.

**Что соответствует в книге:** Прямая реализация формулы `W ≈ U * S * V^T` из главы 2. Почему `svds`, а не `svd`? Потому что `svds` работает с разреженными (sparse) матрицами и вычисляет только k наибольших сингулярных значений.

**Разбор возвращаемого значения:**
- `np.diag(sigma)` превращает вектор сингулярных значений в диагональную матрицу S (k x k)
- `np.dot(np.diag(sigma), vt)` это произведение S * V^T (k x число_слов)
- `.T` транспонирует результат, получаем (число_слов x k)

Каждая строка результата это эмбеддинг одного слова размерности k.

**Что мы меняем:** Ничего принципиального. Функция работает с любой матрицей W. Только аккуратно подбираем k.

#### Функция create_dictionary()

**Что делает в ноутбуке:** Принимает список слов и матрицу SVD-эмбеддингов. Строит словарь Python: ключ это строка (слово), значение это numpy-вектор длины k (SVD-эмбеддинг).

**Важное свойство SVD-эмбеддингов:** Если у нас есть эмбеддинг размерности k, мы можем бесплатно получить эмбеддинг любой меньшей размерности m <= k: просто берем первые m компонент. Это работает потому, что сингулярные значения упорядочены по убыванию. У FastText, Word2Vec и BERT такого свойства нет.

#### Функция convert_text_to_vector()

**Что делает в ноутбуке:** Превращает текст в набор числовых векторов для подачи в нейросеть. Скользящим окном размера n проходит по словам. Для каждого n-gram берет эмбеддинги всех n слов (первые m компонент). Конкатенирует их в один вектор длины n * m.

**Что мы меняем:** Подаем не сырой текст, а лемматизированный (после CLASSLA).

#### Сводная таблица: ноутбук -> теория -> наш проект

| Функция ноутбука | Глава книги | Что в нашем проекте |
|---|---|---|
| `make_matrix_W_list_of_words()` | Гл. 2, раздел 2.1 (матрица W) | BM25-матрица вместо TF-IDF, леммы от CLASSLA |
| `apply_svd()` | Гл. 2, раздел 2.2 (SVD) | Без изменений, подбираем k для 98 документов |
| `create_dictionary()` | Гл. 2, рис. 2.2 (отображение M -> L) | Словарь "лемма -> SVD-вектор" для македонского |
| `convert_text_to_vector()` | Гл. 3, раздел 3.2 (расширение) | n-gram эмбеддинги как вход для LSTM |

### 1.4. Гиперпараметры SVD для нашего македонского корпуса

#### Ранг k (размерность эмбеддингов)

**Рекомендация: начать с k = 50, диапазон для экспериментов 30-97.**

Обоснование:

1. **Ограничение сверху.** Ранг k не может быть больше min(число_документов, число_слов). У нас 98 документов, значит k < 98. Реалистичный потолок: k = 97.

2. **Рекомендации из книги.** Bellegarda пишет, что типичные значения R лежат в диапазоне 100-1000. Но это для корпусов в десятки тысяч документов. Наш корпус маленький (98 документов), поэтому верхнюю границу надо сдвинуть вниз.

3. **Правило для маленьких корпусов.** Чем меньше документов, тем меньше независимых "направлений смысла" можно из них извлечь. Для 98 документов разумно брать k от 30 до 97. Слишком маленький k (< 30) потеряет много информации. Слишком большой (близко к 98) не будет сжимать и не уберет шум.

4. **Практический выбор.** Начинаем с k = 50, потом пробуем k = 30, 70, 80, 90. Смотрим на сингулярные значения: строим scree plot и ищем "колено" (точку, после которой значения резко падают).

#### min_df (минимальная частота документов)

**Рекомендация: min_df = 2 или min_df = 3.**

`min_df` отсекает слова, которые встречаются слишком редко. Слово, которое есть только в одном документе из 98, не дает информации о связях между документами. Для SVD такие слова это просто шум.

#### max_df (максимальная частота документов)

**Рекомендация: max_df = 0.85.**

`max_df` отсекает слова, которые встречаются почти везде (стоп-слова, союзы, частицы). При 98 документах это ~83 документа. Если стоп-слова удалены заранее, можно поднять до 0.90.

#### Параметры BM25

- **k1 = 1.5** (насыщение частоты). Диапазон: 1.2-2.0
- **b = 0.75** (нормализация по длине документа). Для текстов разной длины это важно. Диапазон: 0.5-0.75

#### Сводная таблица гиперпараметров

| Параметр | Значение | Диапазон | Обоснование |
|---|---|---|---|
| k (ранг SVD) | 50 (старт) | 30-97 | Ограничен числом документов (98). Колено scree plot |
| min_df | 2 | 2-3 | Убрать слова-уникумы, оставить осмысленные леммы |
| max_df | 0.85 | 0.80-0.90 | Убрать слишком частые слова |
| BM25 k1 | 1.5 | 1.2-2.0 | Насыщение частоты термина |
| BM25 b | 0.75 | 0.5-0.75 | Нормализация по длине документа |

#### Как подбирать k на практике

1. Построить BM25-матрицу W с выбранными min_df и max_df
2. Запустить SVD с k_max = 97
3. Построить scree plot сингулярных значений
4. Найти "колено" графика
5. Попробовать 3-4 значения k вокруг колена
6. Оценить качество (coherence score кластеров слов или качество генерации LSTM)
7. Выбрать k, дающий лучший результат

**Замечание о размерности m для LSTM:** В функции `convert_text_to_vector()` параметр m определяет, сколько компонент эмбеддинга брать. Можно взять m < k. Стратегия: сначала вычислить SVD с максимальным k (97), потом экспериментировать с разными m (20, 50, 80, 97), не пересчитывая SVD.

## Часть 2. Теоретическая база: сетевой анализ и графы

### 2.1. Конспект книги Barrat, Barthelemy, Vespignani (главы 1-3)

Книга посвящена динамическим процессам на сложных сетях. Первые три главы закладывают фундамент: базовые понятия теории графов, свойства реальных сетей и модели сетей.

#### Глава 1. Предварительные сведения: сети и графы

**Что такое сеть**

Сеть (network) -- любая система, которую можно представить в виде графа. Узлы (vertices, nodes) -- элементы системы, ребра (edges, links) -- связи между ними. В нашем проекте: вершины -- это концепты (существительные с модификаторами из македонского текста), ребра -- это отношения между ними (глаголы, предлоги, притяжание).

**Графы и подграфы**

- **Неориентированный граф** G = (V, E): V -- множество вершин, E -- множество неупорядоченных пар вершин
- **Ориентированный граф** (digraph): ребра имеют направление
- **Матрица смежности** (adjacency matrix): квадратная матрица N x N, x_ij = 1 если есть ребро
- **Плотность** (density): D = E / [N(N-1)/2] -- отношение существующих ребер к максимально возможным
- **Клика** (clique): полный подграф, где каждая пара вершин соединена
- **Сообщества** (communities): подграфы с плотными внутренними и слабыми внешними связями

**Пути и связность**

- **Путь** (path): последовательность вершин и ребер от одной вершины к другой
- **Связный граф**: из любой вершины можно добраться до любой другой
- **Компонента связности**: максимальный связный подграф
- **Гигантская компонента** (giant component): наибольшая компонента, размер которой растет пропорционально N
- Для ориентированных графов различают слабо связную (WCC) и сильно связную (SCC) компоненты

**Кратчайший путь и диаметр**

- **Диаметр**: d_G = max(l_ij) -- максимальное расстояние между любыми двумя вершинами
- **Средний кратчайший путь**: `<l> = (1 / N(N-1)) * sum(l_ij)`
- "Эффект маленького мира" (small-world): любая пара вершин соединена коротким путем (`<l> ~ log(N)`)

**Степень вершины и центральности**

- **Степень** (degree): количество ребер вершины. Для ориентированных: in-degree + out-degree
- **Degree centrality**: нормализованная степень. Много связей = важный узел
- **Closeness centrality**: обратная средняя длина путей. "Близость" ко всем остальным
- **Betweenness centrality**: доля кратчайших путей, проходящих через вершину. "Мосты" между частями графа
- **Eigenvector centrality**: рекурсивная мера -- вершина важна, если ее соседи тоже важны

**Кластеризация**

Коэффициент кластеризации C(i) = e_i / (k_i(k_i-1)/2) -- показывает, насколько соседи вершины связаны друг с другом. Если C = 1, все соседи связаны (клика). Если C = 0, ни одна пара соседей не связана.

**Статистические характеристики**

- **Распределение степеней** P(k) -- для реальных сетей часто степенной закон (power law)
- **Ассортативность** (assortativity): r > 0 -- hubs соединяются с hubs, r < 0 -- hubs с мелкими вершинами
- **Rich-club**: тенденция вершин с высокой степенью образовывать плотные связи

**Взвешенные сети**

- **Вес ребра** w_ij -- интенсивность связи
- **Сила вершины** (vertex strength): s_i = sum(w_ij) -- обобщение степени с учетом весов

#### Глава 2. Сети и сложность

Реальные сети (социальные, транспортные, интернет, биологические) обладают двумя ключевыми свойствами:

1. **Маленький мир + высокая кластеризация**: средний путь `<l> ~ log(N)`, но соседи часто связаны друг с другом. Случайные графы (Erdos-Renyi) имеют маленький мир, но низкую кластеризацию.

2. **Гетерогенность и тяжелые хвосты**: распределение степеней `P(k) ~ k^(-gamma)` (gamma от 2 до 3). Большинство вершин имеют мало связей, но немногие "хабы" имеют очень много. Такие сети называются **scale-free** (безмасштабные).

#### Глава 3. Модели сетей

- **Erdos-Renyi** G(N, p): каждая пара соединяется с вероятностью p. Распределение Пуассона, маленький мир, но нет кластеризации и нет scale-free.

- **Watts-Strogatz** (Small-World): кольцо + случайные "перебросы" ребер. Совмещает маленький мир и высокую кластеризацию. Достаточно совсем немного случайности (shortcut-ребер), чтобы резко сократить расстояния.

- **Barabasi-Albert** (Preferential Attachment): новые вершины чаще присоединяются к уже хорошо связанным. "Богатые становятся еще богаче". Результат: степенной закон P(k) ~ k^(-3).

- **Модели копирования**: новая вершина копирует связи прототипа с вероятностью alpha. Неявно реализует preferential attachment.

#### Ключевые формулы

| Метрика | Формула | Что показывает |
|---------|---------|----------------|
| Density | D = 2E / (N(N-1)) | Доля существующих ребер от максимума |
| Clustering C(i) | e_i / (k_i(k_i-1)/2) | Связность соседей вершины |
| Closeness centrality | 1 / sum(l_ij) | Близость вершины ко всем остальным |
| Betweenness centrality | sum(sigma_hj(i)/sigma_hj) | Доля кратчайших путей через вершину |
| Degree distribution | P(k) ~ k^(-gamma) | Вероятность степени k (для scale-free) |
| Average shortest path | `<l>` ~ log(N)/log(`<k>`) | Типичное расстояние между вершинами |
| Assortativity r | Pearson по степеням концов ребер | Склонность к связи с похожими вершинами |
| Vertex strength | s_i = sum(w_ij) | Сумма весов ребер вершины |

### 2.2. Связь сетевого анализа с юмором

#### Семантический граф текста -- это сеть

Когда мы строим семантический граф из текста на македонском языке, мы получаем сеть:
- **Вершины** -- концепты: существительные с их модификаторами. Например: "голем куче" (большая собака), "мал парк" (маленький парк), "стар човек" (старый человек).
- **Ребра** -- отношения между концептами: глаголы ("трча" -- бежит), предлоги ("во" -- в, "на" -- на), притяжание ("неговиот" -- его).

Все метрики из книги (degree, clustering, centrality, path length) работают и здесь. Но мы ищем не маршруты самолетов, а структуру смысла в тексте.

#### Теория юмора: semantic incongruity

Одна из самых влиятельных теорий юмора -- **теория несоответствия** (incongruity theory). Суть: юмор возникает, когда мы ожидаем одно, а получаем совсем другое. Шутка "срабатывает", когда два понятия, которые обычно далеки друг от друга, вдруг оказываются связаны.

Пример: "Доктор сказал мне, что я должен перестать устраивать ужины на четверых. Если только за столом нет трех других людей." Ожидание -- медицинская рекомендация. Реальность -- комментарий об одиночестве. Два далеких контекста сталкиваются.

В терминах графа: юмор -- это **ребро между семантически далекими вершинами**. "Далекие" значит -- с низким cosine similarity между их embedding-ами.

Пример ближе к нашему проекту: "Мачката седи на компјутерот и ги брише фајловите" (Кошка сидит на компьютере и удаляет файлы). "Мачка" (кошка) и "фајлови" (файлы) -- семантически далекие концепты. Неожиданная связь между ними -- источник юмора.

#### Как метрики графа измеряют "неожиданность"

**Cosine distance между embedding-ами вершин.** Каждая вершина имеет FastText embedding (300-мерный вектор). Косинусное расстояние показывает семантическую далекость. **Semantic incongruity score** графа -- среднее косинусное расстояние по всем ребрам. Чем выше, тем больше потенциал для юмора.

**Betweenness centrality и "мосты юмора".** Вершина с высоким betweenness -- "мост" между разными частями графа. В юмористическом тексте такие мосты связывают два неожиданных контекста. Если вершина имеет высокий betweenness, но низкую степень -- она ключевой связующий элемент.

**Clustering coefficient и плотность окрестности.** Низкий коэффициент кластеризации: соседи вершины не связаны друг с другом. Это "звезда", которая связывает несвязанные концепты -- потенциальная "точка юмора".

**Degree distribution и "хабы юмора".** Интересны хабы с высоким semantic incongruity и вершины с аномально высокой степенью в неожиданных контекстах.

**Average shortest path.** Перекликается с моделью Watts-Strogatz: несколько "перекинутых" ребер между далекими частями графа (shortcut-ы) резко сокращают средний путь. В юмористическом тексте эти shortcut-ы и есть неожиданные связи.

#### Как юмор выглядит в графе

| Свойство графа | В обычном тексте | В юмористическом тексте |
|---------------|------------------|------------------------|
| Semantic incongruity score | Низкий: связи между близкими концептами | Высокий: связи между далекими концептами |
| Betweenness centrality | Равномерно распределена | Несколько "мостов" между далекими кластерами |
| Clustering coefficient | Умеренный: соседи связаны | Низкий: соседи не связаны друг с другом |
| Degree distribution | Плавное, ожидаемые хабы | Аномальные хабы в неожиданных позициях |
| Edge weights + distance | Частые ребра между близкими словами | Редкие ребра между далекими словами |
| Shortest path | Определяется тематикой | Сокращен "shortcut-ами" юмора |

### 2.3. Метрики графа для нашего проекта

Какие графовые метрики уже реализованы в `metrics_mk.py`, какие стоит добавить.

#### Что уже реализовано

Функция `calculate_metrics_mk(graph)` возвращает словарь из 16 метрик:

| Метрика | Ключ в словаре | Что считает |
|---------|---------------|-------------|
| Число вершин | `num_vertices` | Сколько концептов в графе |
| Число ребер | `num_edges` | Сколько отношений в графе |
| Средняя степень | `average_degree` | 2E/N |
| Средняя входящая степень | `average_in_degree` | Среднее k_in |
| Средняя исходящая степень | `average_out_degree` | Среднее k_out |
| Плотность | `density` | Доля существующих ребер от максимума |
| Средний clustering coefficient | `average_clustering_coefficient` | Средняя кластеризация |
| Ассортативность | `assortativity` | Корреляция степеней на концах ребер |
| Число слабо связных компонент | `weakly_connected_components` | Сколько изолированных частей |
| Размер гигантской компоненты | `giant_component_size` | Вершин в наибольшей компоненте |
| Средний кратчайший путь | `average_shortest_path_length` | Средняя длина кратчайшего пути |
| Диаметр | `diameter` | Максимальное расстояние |
| 4 средних центральности | `average_*_centrality` | degree, betweenness, closeness, eigenvector |

Плюс 4 функции для индивидуальных центральностей, 3 функции для распределений (степеней, кластеризации, путей), текстовые метрики (TTR, MTLD, T-score, длина слов, слоги) и метрики юмора (`semantic_incongruity_score`, `lexical_surprise_mk`).

#### Что полезно добавить

**Для анализа юмора:**
1. **Edge-level semantic incongruity** -- cosine distance для каждого ребра отдельно. Позволит найти конкретные "юмористические" связи
2. **Incongruity variance** -- дисперсия cosine distance по ребрам. Высокая дисперсия = контраст между нормальными и неожиданными связями

**Для community detection:**
3. **Community detection** через `networkx.community` (greedy modularity, Louvain). Сообщества = тематические кластеры, ребра между ними = потенциальные юмористические мосты
4. **Modularity** -- качество разбиения на сообщества

**Для описания структуры:**
5. **Power law fit** -- проверить, подчиняется ли распределение степеней степенному закону
6. **Rich-club coefficient** -- тенденция хабов образовывать плотные связи

#### Приоритеты добавления

| Приоритет | Метрика | Зачем нужна |
|-----------|---------|-------------|
| 1 | Community detection + modularity | Features для нейросети |
| 2 | Edge-level incongruity | Таргетирование юмористических ребер |
| 3 | Power law fit | Характеристика графа целиком |
| 4 | Incongruity variance | Различение юмористических и обычных текстов |

### 2.4. Как графовые метрики становятся фичами для нейросети

Вся суть: берем семантический граф текста, считаем числовые метрики, подаем их на вход нейросети.

#### Три уровня features

**Уровень 1: Features вершины (~310 чисел)**

| Тип | Состав | Размерность |
|-----|--------|-------------|
| Embedding | FastText вектор из cc.mk.300.bin | 300 |
| Графовые метрики | degree, in_degree, out_degree, clustering, betweenness, closeness | 6 |
| Контекстные | mean_edge_incongruity, lexical_surprise, community_id | 2-3 |

Конкатенация: `[embedding_300d, degree, in_degree, out_degree, clustering, betweenness, closeness, mean_incongruity, surprise, community_id]`

**Уровень 2: Features ребра (~305 чисел)**

| Тип | Состав | Размерность |
|-----|--------|-------------|
| Embedding | FastText вектор слова-отношения | 300 |
| Числовые | weight, cosine_distance, t_score, degree_diff, edge_betweenness | 3-5 |

**Уровень 3: Features графа целиком (~25 чисел)**

- Структурные: num_vertices, num_edges, density, average_degree, clustering, assortativity, components, giant_component_ratio, shortest_path, diameter (10-12)
- Центральности: 4 средних (degree, betweenness, closeness, eigenvector)
- Метрики юмора: semantic_incongruity, incongruity_variance, lexical_surprise, modularity (3-4)
- Текстовые: TTR, MTLD, average_word_length, syllables_per_word (4-5)

#### Как подавать features в разные архитектуры

- **Вариант 1 (GNN + LSTM):** GNN получает features вершин и ребер напрямую, LSTM получает features графа как initial state
- **Вариант 2 (SVD + LSTM):** Features графа конкатенируются с SVD-вектором документа
- **Вариант 3 (BM25 + LSTM):** Features графа + embedding-и вершин подаются в LSTM

#### Конкретный пример

Текст: "Кучето трча низ паркот и лае на мачката."
Граф: вершины {куче, парк, мачка}, ребра {куче --трча--> парк, куче --лае--> мачка}

Features вершины "куче":
```
embedding = [0.12, -0.34, 0.56, ...] (300 чисел из FastText)
degree = 2, in_degree = 0, out_degree = 2
clustering = 0.0    (парк и мачка не связаны)
betweenness = 1.0   (все пути через куче)
mean_incongruity = 0.45
```

Features графа:
```
num_vertices = 3, num_edges = 2, density = 0.33
semantic_incongruity = 0.45, lexical_surprise = 9.2
```

#### Что features дают нейросети

Без графовых features нейросеть видит только последовательность слов. С ними она получает:
- **Degree** -- насколько "центральный" концепт
- **Clustering** -- насколько изолированная связь
- **Betweenness** -- "мостовой" ли концепт
- **Incongruity** -- насколько неожиданна связь
- **Community** -- к какой тематической группе принадлежит

Для генерации юмора это критично: нейросеть может научиться "когда incongruity высокий и clustering низкий, это смешно".

Все features нормализуются перед подачей (MinMaxScaler или StandardScaler из sklearn), чтобы метрики с разным масштабом (degree = 15, density = 0.03) влияли равноценно.

## Часть 3. Анализ вариантов архитектуры

Описываем три варианта архитектуры нейросети, сравниваем их по ключевым критериям, и обосновываем выбор основного варианта для реализации. Также описываем baseline-модели (цепь Маркова, n-gram) и архитектуру моделирования юмора.

### Вариант 1: Graph-based + Neural Hybrid (GNN encoder + LSTM decoder)

Первый вариант архитектуры -- самый амбициозный. Идея: закодировать семантический граф текста с помощью графовой нейросети (GNN), получить один вектор, описывающий весь граф, и скормить этот вектор LSTM-декодеру, который сгенерирует текст.

#### 3.1. Общая схема варианта 1

### Поток данных

Текст на македонском языке проходит через несколько этапов обработки, прежде чем из него получится новый (юмористический) текст. Вот весь путь:

```
  Македонский текст (сырой)
        |
        v
  CLASSLA (tokenize + POS + lemma)
  + spaCy mk_core_news_lg (depparse)
        |
        v
  make_graph_mk.py
  (построение семантического графа)
        |
        v
  Граф G = (V, E)
  вершины с FastText-эмбеддингами (300-dim),
  ребра со значениями (глаголы, предлоги)
        |
        v
  convert_to_networkx()      <-- метод класса Graph
        |
        v
  nx.DiGraph                 <-- граф в формате NetworkX
        |
        v
  from_networkx()            <-- конвертация torch_geometric
        |
        v
  torch_geometric.data.Data  <-- граф в формате PyTorch Geometric
        |
        v
  Graph Encoder: GNN
  (GCN или GAT, 2-3 слоя)
        |
        v
  Вектор графа h_G (256-dim)
  (один вектор на весь граф)
        |
        v
  Text Decoder: LSTM с attention
        |
        v
  Сгенерированный текст на македонском
```

### Что происходит на каждом этапе

**CLASSLA + spaCy.** Два NLP-инструмента работают вместе: CLASSLA делает токенизацию, POS-тегирование и лемматизацию, а spaCy (модель `mk_core_news_lg`) -- dependency parsing (разбор синтаксических зависимостей). Результат: для каждого предложения мы знаем, какие слова как связаны.

**make_graph_mk.py.** Берет результат NLP-обработки и строит семантический граф. Вершины -- концепты (существительные с модификаторами, например "голем куче" -- большая собака). Ребра -- отношения (глаголы, предлоги, притяжание). Используются классы Vertex, Edge, Graph из `rulebased-concept-tree-main/graph/`, которые подтверждены как языконезависимые.

**convert_to_networkx().** Метод класса Graph, который конвертирует наш граф в формат NetworkX (nx.DiGraph). Это стандартный формат для работы с графами в Python.

**from_networkx().** Функция из библиотеки torch_geometric, которая конвертирует nx.DiGraph в `torch_geometric.data.Data` -- формат, понятный графовым нейросетям в PyTorch. Конвертация стандартная и занимает пару строк кода.

**Graph Encoder (GNN).** Графовая нейросеть обрабатывает граф и выдает один вектор фиксированного размера (256 чисел), который описывает весь граф. Подробнее -- в разделе 3.2.

**Text Decoder (LSTM).** Получает вектор графа и генерирует текст токен за токеном. Подробнее -- в разделе 3.3.

### Аналогия из жизни

Представьте, что вы пересказываете друзьям прочитанную книгу. Сначала вы "строите" в голове карту персонажей и событий (это граф). Потом "сжимаете" эту карту до общего впечатления (это GNN encoder -- получаем вектор графа). И наконец, основываясь на этом впечатлении, рассказываете историю своими словами (это LSTM decoder).

#### 3.2. Graph Encoder: как превратить граф в один вектор

Graph Encoder -- это графовая нейросеть (GNN, Graph Neural Network). Она принимает на вход граф с эмбеддингами вершин и выдает один вектор, описывающий весь граф.

Рассмотрим два подварианта: GCN и GAT.

### Подвариант A: GCN (Graph Convolutional Network)

GCN -- Graph Convolutional Network, графовая свёрточная сеть. Работает по простому принципу: каждая вершина "опрашивает" своих соседей и обновляет свой эмбеддинг на основе их эмбеддингов.

**Как устроен один слой GCN:**

Для каждой вершины v берем эмбеддинги всех ее соседей, усредняем их (с поправкой на степени вершин) и пропускаем через нелинейную функцию ReLU.

Формула одного слоя:

```
h_v^(l+1) = ReLU( SUM( h_u^(l) / sqrt(deg(v) * deg(u)) ) )
```

Где:
- `h_v^(l)` -- эмбеддинг вершины v на слое l (на слое 0 это исходный FastText-эмбеддинг)
- `h_u^(l)` -- эмбеддинг соседа u на слое l
- `deg(v)` -- степень вершины v (сколько у нее соседей)
- `deg(u)` -- степень соседа u
- `sqrt(deg(v) * deg(u))` -- нормализация, чтобы вершины с большим количеством соседей не "перекрикивали" остальных
- SUM -- сумма по всем соседям u вершины v
- ReLU -- функция активации: `ReLU(x) = max(0, x)`

**Что происходит за несколько слоев:**

- После 1 слоя каждая вершина "знает" о своих непосредственных соседях
- После 2 слоев -- о соседях соседей
- После 3 слоев -- о соседях соседей соседей

Для наших графов (обычно 10-50 вершин) достаточно 2-3 слоев, чтобы каждая вершина "увидела" весь граф.

**Как получить один вектор на весь граф:**

После всех слоев GCN у нас есть обновленные эмбеддинги всех вершин. Нужно "сжать" их в один вектор. Два способа:

- **Mean pooling** -- берем среднее по всем вершинам: `h_G = mean(h_v для всех v)`
- **Max pooling** -- берем поэлементный максимум: `h_G = max(h_v для всех v)`

Mean pooling лучше, когда важен "общий тон" графа. Max pooling -- когда важны самые яркие вершины.

**Аналогия:** GCN -- это как опрос общественного мнения. Каждый человек спрашивает у соседей, что они думают, и корректирует свое мнение. После нескольких раундов каждый человек учитывает мнения далеких людей (через цепочку соседей). В конце мы берем среднее мнение всех -- это и есть "вектор графа".


### Подвариант B: GAT (Graph Attention Network)

GAT -- Graph Attention Network, графовая сеть с вниманием. Работает похоже на GCN, но с важным отличием: при "опросе" соседей каждому соседу присваивается свой вес (attention weight). Более важные соседи влияют сильнее.

**Формула одного слоя GAT:**

```
h_v^(l+1) = ReLU( SUM( alpha_vu * W * h_u^(l) ) )
```

Где:
- `W` -- обучаемая матрица весов (одна на все вершины)
- `alpha_vu` -- attention weight: насколько сосед u важен для вершины v
- Остальное -- как в GCN

**Как вычисляются attention weights:**

```
alpha_vu = softmax( LeakyReLU( a^T * [W*h_v || W*h_u] ) )
```

Где:
- `a` -- обучаемый вектор (параметры attention)
- `[... || ...]` -- конкатенация двух векторов
- softmax -- нормализация, чтобы все alpha_vu по соседям u суммировались в 1
- LeakyReLU -- вариант ReLU, который пропускает небольшие отрицательные значения

Звучит сложно, но суть простая: нейросеть сама учится определять, какие соседи важнее, а какие -- нет.

**Почему GAT лучше для семантических графов:**

В нашем графе связи не равнозначны. Ребро "куче --лае--> мачка" (собака лает на кошку) гораздо информативнее, чем ребро "куче --во--> парк" (собака в парке). GCN усредняет всех соседей одинаково, а GAT может научиться уделять больше внимания "интересным" соседям (тем, с которыми связь неожиданная).

**Multi-head attention:** на практике в GAT используют несколько "голов" attention (обычно 4 или 8). Каждая голова -- свой набор весов alpha_vu. Результаты всех голов конкатенируются или усредняются. Это как спросить 4 разных эксперта -- каждый обращает внимание на свое, и вместе они дают более полную картину.

**Pooling** -- тот же, что и для GCN: mean или max pooling по обновленным эмбеддингам вершин.

**Аналогия:** GAT -- это как опрос экспертов, где вы знаете, кто из них разбирается в теме лучше. Вместо того чтобы усреднять мнения всех (как в GCN), вы прислушиваетесь к тем, кто говорит по делу (высокий attention weight), и меньше слушаете тех, кто говорит общие слова (низкий attention weight).


### Сравнение GCN и GAT

| Свойство | GCN | GAT |
|---|---|---|
| Усреднение соседей | Равномерное (с поправкой на степень) | Взвешенное (attention weights) |
| Обучаемые параметры | Меньше | Больше (attention + multi-head) |
| Качество для семантических графов | Хорошее | Лучше (учитывает важность связей) |
| Скорость обучения | Быстрее | Медленнее |
| Риск переобучения на 98 текстах | Высокий | Очень высокий |
| Реализация в PyTorch Geometric | `GCNConv` | `GATConv` |

**Рекомендация для нашего проекта (если выбираем вариант 1):** начать с GCN (проще, меньше параметров, меньше риск переобучения). Если GCN покажет приемлемый результат -- попробовать GAT и сравнить.

#### 3.3. Text Decoder: как сгенерировать текст из вектора графа

Text Decoder -- это LSTM-сеть, которая принимает вектор графа h_G (256 чисел) и генерирует текст токен за токеном.

### Архитектура декодера

```
h_G (256-dim)
    |
    v
initial hidden state h_0 = h_G
initial cell state c_0 = Linear(h_G)
    |
    v
LSTM (1-2 слоя, hidden_size=256)
    |
    v
на каждом шаге t:
  вход: embedding предыдущего токена x_t (300-dim FastText или обучаемый)
  выход: h_t (256-dim)
    |
    v
Linear (256 -> vocab_size)
    |
    v
softmax -> вероятности следующего токена
    |
    v
выбираем следующий токен
```

### Как работает генерация

1. Вектор графа h_G (256 чисел) подается как начальное скрытое состояние (initial hidden state) LSTM. Это как "задать тему" для генерации: LSTM начинает работу, уже "зная" структуру графа.

2. На первом шаге LSTM получает специальный стартовый токен `<BOS>` (Beginning Of Sequence). Выдает вероятности для первого слова текста.

3. На каждом следующем шаге LSTM получает предыдущий токен и выдает вероятности для следующего. Процесс продолжается, пока не будет сгенерирован токен `<EOS>` (End Of Sequence) или не достигнута максимальная длина.

4. Вероятности вычисляются через линейный слой (Linear, hidden_size -> vocab_size) и softmax. Softmax превращает числа в вероятности: все значения от 0 до 1, в сумме дают 1.

### Teacher forcing при обучении

При обучении есть хитрость -- teacher forcing. Вместо того чтобы подавать сгенерированный токен на следующий шаг, мы подаем правильный токен из обучающей выборки.

Зачем? Если подавать сгенерированные токены, одна ошибка на раннем шаге приводит к лавине ошибок на всех последующих шагах. LSTM "сбивается с пути" и генерирует чепуху. Teacher forcing не дает ошибкам накапливаться.

Аналогия: представьте, что ученик пишет диктант. Без teacher forcing -- учитель диктует следующее слово только после того, как ученик написал предыдущее (даже с ошибкой). С teacher forcing -- учитель показывает правильное написание каждого слова сразу после того, как ученик попытался.

На практике часто используют scheduled sampling: в начале обучения teacher forcing используется почти всегда (вероятность 90-100%), а к концу обучения вероятность снижается (до 50-60%). Это помогает LSTM научиться исправлять собственные ошибки.

### Стратегии генерации (sampling)

При генерации (не при обучении) нужно выбирать следующий токен из распределения вероятностей. Есть несколько стратегий:

**Greedy search:** всегда берем самый вероятный токен. Просто, но тексты получаются скучными и повторяющимися.

**Top-k sampling:** берем k самых вероятных токенов, нормализуем их вероятности (чтобы суммировались в 1) и случайно выбираем один из них. Типичные значения k: 10-50. Чем больше k, тем разнообразнее тексты, но выше риск бессмыслицы.

**Temperature sampling:** делим logits (числа до softmax) на temperature T перед softmax:
- T = 1.0 -- стандартное поведение
- T < 1.0 (например, 0.5) -- распределение становится "острее": вероятные токены становятся еще вероятнее, маловероятные -- еще менее вероятны. Тексты более предсказуемые
- T > 1.0 (например, 1.5) -- распределение "размазывается": все токены становятся более равновероятными. Тексты более разнообразные (но и более хаотичные)

**Для юмора** лучше подходит умеренно высокая температура (1.1-1.5) или top-k с k = 20-40. Юмор требует неожиданности, а слишком "правильные" тексты (greedy/low temperature) получаются скучными.

#### 3.4. Оценка сложности варианта 1

### Плюсы

- **Явно учитывает графовую структуру.** GNN "видит" связи между концептами, их вложенность, кластеры. Для юмора это важно: GNN может научиться распознавать "неожиданные" связи (ребра между семантически далекими вершинами) и использовать их при генерации.

- **Modern deep learning подход.** GNN + LSTM -- это хорошо изученная архитектура, по которой много статей и примеров кода. Преподавателю будет видно, что студент владеет актуальными методами.

- **Масштабируемость.** Если в будущем появится больше текстов (не 98, а 1000), эта архитектура справится без принципиальных изменений.

- **Интерпретируемость через attention.** Если используем GAT, attention weights показывают, какие связи в графе модель считает важными. Это можно визуализировать и показать преподавателю.

### Минусы

- **Нужна библиотека torch_geometric.** Этой библиотеки нет в текущих зависимостях проекта. Установка нетривиальна: torch_geometric зависит от конкретной версии PyTorch и CUDA, и на некоторых системах установка вызывает проблемы. Нужно добавить `torch-geometric`, `torch-scatter`, `torch-sparse` в requirements.txt.

- **Сложная отладка.** У GNN-архитектуры много движущих частей: конвертация графа в torch_geometric формат, GNN encoder, pooling, LSTM decoder, attention. Если что-то идет не так (loss не уменьшается, генерируется мусор), трудно понять, в какой компоненте проблема.

- **Высокий риск переобучения.** У нас только 98 текстов. GNN + LSTM -- это тысячи обучаемых параметров. Нейросеть может просто запомнить все 98 текстов наизусть, не научившись обобщать. Для борьбы с переобучением нужны: dropout, early stopping, data augmentation -- но для 98 текстов даже эти приемы могут не помочь.

- **Время обучения.** На CPU (без GPU) обучение GNN + LSTM может занять часы. С GPU -- минуты, но GPU может не быть на рабочей машине.

- **Конвертация графа в torch_geometric.** Хотя `from_networkx()` стандартная, нужно аккуратно подготовить атрибуты вершин и ребер: все features должны быть тензорами одинаковой размерности. Если у одной вершины эмбеддинг есть, а у другой нет (OOV-слово без FastText), это вызовет ошибку.

### Объем кода

Примерно 600-800 строк Python-кода:
- Конвертация Graph -> torch_geometric.data.Data: ~100 строк
- GNN Encoder (GCN или GAT + pooling): ~150 строк
- LSTM Decoder с attention: ~200 строк
- Обучающий цикл (train loop, loss, optimizer): ~150 строк
- Генерация (inference с sampling): ~100 строк

### Риск для курсовой: ВЫСОКИЙ

Если все сработает, результат впечатляющий. Но вероятность того, что нейросеть не сойдется или переобучится на 98 текстах, существенная. На отладку может уйти непредсказуемо много времени.

### Итоговая оценка варианта 1

Этот вариант -- самый технически продвинутый, но и самый рискованный. Он подходит, если есть запас времени на отладку и если преподаватель ценит амбициозность подхода. Если важнее надежность и предсказуемость результата, лучше рассмотреть варианты 2 или 3.

### Вариант 2: SVD/LSA + генеративная модель (LSTM)

Второй вариант архитектуры -- прагматичный и хорошо обоснованный теоретически. Идея: берем корпус лемматизированных текстов, строим BM25-матрицу, применяем SVD-разложение (как описано в конспекте книги LSM, шаг 1), получаем компактные эмбеддинги слов, и обучаем LSTM генерировать текст в этом латентном пространстве.

#### 4.1. Общая схема варианта 2

### Поток данных

```
  Корпус: 98 отфильтрованных текстов на македонском
  (лемматизированные, из classla_tokens в nlp_data.db)
        |
        v
  BM25-матрица W
  (98 документов x ~10 000 лемм)
        |
        v
  SVD-разложение: W = U * Sigma * V^T
  (ранг k, стартовое значение k=50, диапазон 30-97)
        |
        v
  U (98 x k)         -- эмбеддинги документов
  Sigma (k,)         -- веса измерений
  V^T (k x ~10000)   -- координаты слов
        |
        v
  Словарь эмбеддингов: лемма -> k-мерный вектор
  (каждая лемма i: вектор = Sigma * V^T[:, i])
        |
        v
  LSTM-генератор
  (embedding layer = SVD-эмбеддинги, frozen)
        |
        v
  Сгенерированный текст на македонском
```

### Почему ранг k ограничен числом 97

В шаге 1 (конспект LSM, раздел 1.4) мы определили: ранг SVD не может превышать min(число_документов, число_слов) - 1. У нас 98 документов и примерно 10 000 лемм. Значит k < 98, то есть максимум k = 97.

В книге Bellegarda типичные значения R (его обозначение для k) -- от 100 до 1000, но для корпусов в десятки тысяч документов. Наш корпус маленький, поэтому стартуем с k = 50 и экспериментируем в диапазоне 30-97.

### Отличие от варианта 1

В варианте 1 (GNN + LSTM) мы кодируем граф нейросетью. Здесь мы используем классическую линейную алгебру (SVD) для получения эмбеддингов и подаем их в LSTM напрямую. Граф явно не участвует в генерации -- вся информация о тексте "упакована" в SVD-эмбеддинги слов.

### Аналогия

Представьте библиотеку с 98 книгами. SVD -- это как составить каталог из 50 "тем", где для каждой книги и каждого слова записано, насколько оно связано с каждой темой. LSTM -- как писатель, который пользуется этим каталогом: зная, какие слова связаны с какими темами, он пишет новый текст, подбирая слова по тематической близости.

#### 4.2. Подготовка данных

Подготовка данных -- самый ответственный этап. От качества BM25-матрицы и SVD-разложения зависит качество эмбеддингов, а от эмбеддингов -- качество генерации.

### Шаг 1: Берем лемматизированные тексты

Тексты уже обработаны в подплане 2. Лемматизированные токены хранятся в таблице `classla_tokens` базы данных `datasets/nlp_data.db`. Столбцы таблицы: `text_id`, `sentence_id`, `token_id`, `form`, `lemma`, `upos`, `xpos`.

Для каждого из 98 текстов собираем все леммы (столбец `lemma`) в один "мешок слов" (bag of words). Фильтруем по POS-тегам: оставляем существительные (NOUN), прилагательные (ADJ), глаголы (VERB), наречия (ADV). Убираем стоп-слова (предлоги, союзы, частицы, местоимения), пунктуацию и числа.

### Шаг 2: Строим BM25-матрицу

BM25-матрица -- это таблица размером 98 (документов) x ~10 000 (лемм), где в каждой ячейке стоит BM25-скор: насколько лемма i важна для документа j.

Формула BM25:

```
BM25(t, d) = IDF(t) * ( tf(t,d) * (k1 + 1) ) / ( tf(t,d) + k1 * (1 - b + b * |d| / avgdl) )
```

Где:
- `tf(t,d)` -- сколько раз лемма t встречается в документе d
- `IDF(t) = log( (N - df(t) + 0.5) / (df(t) + 0.5) )` -- обратная документная частота
- `N` -- общее число документов (98)
- `df(t)` -- в скольких документах встречается лемма t
- `|d|` -- длина документа d (число лемм)
- `avgdl` -- средняя длина документа по корпусу
- `k1 = 1.5` -- контролирует насыщение частоты (слово, встретившееся 10 раз, не в 10 раз важнее, чем 1 раз)
- `b = 0.75` -- контролирует нормализацию по длине документа

В оригинальном ноутбуке `Preprocessing_SVD_example.ipynb` для построения матрицы W используется `TfidfVectorizer` из sklearn. Мы заменяем TF-IDF на BM25-скоры: BM25 лучше учитывает длину документа (параметр b) и насыщение частоты (параметр k1). Для нашего корпуса с текстами разной длины (рассказы vs. романы) это особенно важно.

Фильтрация словаря:
- `min_df = 2` -- лемма должна встречаться минимум в 2 документах (убираем слова-уникумы, опечатки)
- `max_df = 0.85` -- лемма должна встречаться не более чем в 85% документов (убираем слишком частые слова)

Результат: разреженная (sparse) матрица W размером 98 x ~10 000.

### Шаг 3: SVD-разложение

Применяем SVD через `scipy.sparse.linalg.svds`:

```python
from scipy.sparse.linalg import svds

u, sigma, vt = svds(W, k=50)
```

Получаем три компоненты:
- **U** (98 x 50) -- эмбеддинги документов. Каждая строка -- координаты одного текста в 50-мерном пространстве
- **Sigma** (50,) -- вектор сингулярных значений. Показывает "важность" каждого измерения. Первое значение -- самое большое (самый сильный паттерн в данных), последнее -- самое маленькое
- **V^T** (50 x ~10 000) -- координаты слов. Каждый столбец -- координаты одной леммы в 50-мерном пространстве

Важно: `svds` не гарантирует порядок сингулярных значений, поэтому после разложения нужно отсортировать Sigma по убыванию и соответственно переставить строки/столбцы U и V^T. Эта сортировка уже реализована в функции `apply_svd()` из `Preprocessing_SVD_example.ipynb`.

### Шаг 4: Создаем словарь эмбеддингов

Для каждой леммы i вычисляем SVD-эмбеддинг:

```
embedding_i = Sigma * V^T[:, i]
```

Это умножение "взвешивает" координаты леммы: более важные измерения (с большим сингулярным значением) вносят больший вклад. Результат -- вектор длины k (50 чисел при k=50).

Словарь: `{"лемма": numpy-вектор длины k}`. В терминах книги LSM (глава 2) -- это отображение каждого слова в пространство L.

Полезное свойство SVD-эмбеддингов (из конспекта шага 1.3): если мы вычислили эмбеддинги размерности k=50, мы можем бесплатно получить эмбеддинги любой меньшей размерности m <= 50, просто взяв первые m компонент. Не нужно пересчитывать SVD. Это уникальное преимущество SVD перед FastText или Word2Vec.

### Шаг 5: Хранение эмбеддингов через FAISS

**Что такое FAISS:** библиотека от Meta (Facebook AI Similarity Search) для быстрого поиска похожих векторов. Вместо того чтобы сравнивать новый вектор со всеми 10 000 эмбеддингов по очереди, FAISS делает это за O(1) -- практически мгновенно.

**Зачем нам нужен быстрый поиск:** для моделирования юмора через semantic incongruity. Юмор -- это неожиданные связи, то есть ребра между семантически далекими концептами. Чтобы найти пары "далеких" слов, нужно быстро искать не только ближайших, но и самых далеких соседей по cosine similarity. FAISS позволяет делать это эффективно.

**Какой индекс используем:** `faiss.IndexFlatIP` (Inner Product). Это простейший индекс -- точный поиск по скалярному произведению. Для нормализованных векторов скалярное произведение эквивалентно cosine similarity. Создание индекса -- буквально 5 строк кода:

```python
import faiss
import numpy as np

# нормализуем эмбеддинги (чтобы inner product = cosine similarity)
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# создаем индекс
index = faiss.IndexFlatIP(k)  # k -- размерность эмбеддинга
index.add(embeddings_normalized)

# сохраняем на диск
faiss.write_index(index, 'datasets/embeddings.faiss')
```

**Маппинг лемма <-> faiss_id:** FAISS работает с числовыми индексами (0, 1, 2, ...). Нужна таблица соответствия: какая лемма под каким номером. Сохраняем в `nlp_data.db`:

```sql
CREATE TABLE embedding_index (
    lemma TEXT PRIMARY KEY,
    faiss_id INTEGER
);
```

Обратный поиск: по faiss_id -> лемма.

**Pickle как запасной вариант:** для совместимости сохраняем также pickle-копию словаря эмбеддингов в `datasets/svd_embeddings_mk.pkl`. Если FAISS по какой-то причине не установится, можно загрузить эмбеддинги из pickle (но без быстрого поиска).

**Эмбеддинги документов:** матрицу U (98 x k) сохраняем как numpy-файл `datasets/document_embeddings.npy`. Она пригодится для поиска похожих документов и для анализа корпуса.

### Координация с другими подпланами

Важно не путать два набора эмбеддингов:

| Набор | Источник | Размерность | Назначение |
|---|---|---|---|
| **FastText** | cc.mk.300.bin (предобученная модель Facebook) | 300 | Эмбеддинги вершин графа (embedding_manager.py). Используются для cosine distance между концептами, для поиска семантически далеких пар (юмор) |
| **SVD** | BM25-матрица нашего корпуса -> SVD-разложение | k (50) | Embedding layer LSTM-генератора. Используются для генерации текста |

FastText -- для графа и семантической близости. SVD -- для LSTM-генератора.

### Зависимости

- `faiss-cpu` -- нужно добавить в `source/requirements.txt` (пока отсутствует)
- `scipy` -- уже есть (нужна для `svds`)
- `numpy` -- уже есть

#### 4.3. LSTM-генератор на SVD-эмбеддингах

### Архитектура

```
SVD-эмбеддинг токена x_t (k-dim, frozen)
        |
        v
LSTM (1-2 слоя, hidden_size=256)
        |
        v
h_t (256-dim)
        |
        v
Linear (256 -> vocab_size)
        |
        v
softmax -> вероятности следующего токена
        |
        v
выбираем следующий токен -> его SVD-эмбеддинг -> вход на шаг t+1
```

### Embedding layer (frozen)

Первый слой LSTM-генератора -- embedding layer. Обычно этот слой обучаемый (нейросеть сама подбирает эмбеддинги). Но у нас есть готовые SVD-эмбеддинги, и мы используем их как фиксированные (frozen) весовые матрицы.

Почему frozen:
- У нас мало данных (98 текстов). Если позволить обучать embedding layer, нейросеть быстро "забудет" SVD-эмбеддинги и подгонит их под обучающую выборку (переобучение)
- SVD-эмбеддинги уже содержат информацию о связях между словами, полученную из всего корпуса. Обучение может эту информацию разрушить

Технически: создаем `nn.Embedding(vocab_size, k)`, загружаем SVD-эмбеддинги как веса, ставим `requires_grad=False`.

```python
embedding_layer = nn.Embedding(vocab_size, k)
embedding_layer.weight = nn.Parameter(torch.tensor(svd_embeddings), requires_grad=False)
```

Размер embedding layer: vocab_size x k. При vocab_size = 10 000 и k = 50 это 500 000 чисел, но они не обучаются.

### LSTM layer

LSTM (Long Short-Term Memory) -- рекуррентная нейросеть, которая обрабатывает последовательность токенов один за другим, сохраняя "память" о предыдущих токенах.

Параметры:
- **input_size = k** (размерность SVD-эмбеддинга, например 50)
- **hidden_size = 256** (размерность скрытого состояния; 256 -- стандартный выбор, достаточный для наших текстов)
- **num_layers = 1-2** (1 слой проще и быстрее, 2 слоя -- больше емкость модели, но выше риск переобучения)
- **dropout = 0.3** (между слоями, только если num_layers > 1; помогает бороться с переобучением)

Аналогия: LSTM -- как писатель, который пишет текст слово за словом. У него есть "рабочая память" (hidden state, 256 чисел), в которой хранится контекст написанного. После каждого слова память обновляется: что-то забывается (forget gate), что-то добавляется (input gate).

### Output layer

Линейный слой (nn.Linear) преобразует hidden state (256-dim) в вектор размера vocab_size (~10 000). Каждое число -- "скор" для каждого слова словаря. Softmax превращает скоры в вероятности.

```
output = softmax( Linear(h_t) )
```

`output[i]` -- вероятность того, что следующее слово будет i-м словом из словаря.

### Обучение

**Loss function:** cross-entropy loss -- стандартная функция потерь для задач классификации (а выбор следующего слова -- это по сути классификация с vocab_size классами).

```python
loss = nn.CrossEntropyLoss()(output, target)
```

Где `target` -- индекс правильного следующего слова из обучающей выборки.

**Optimizer:** Adam (lr = 0.001). Стандартный выбор для LSTM.

**Teacher forcing:** при обучении на каждом шаге подаем правильный токен (из текста), а не сгенерированный. Это ускоряет обучение и стабилизирует loss.

**Batch size:** маленький (8-16), потому что текстов мало (98).

**Epochs:** 50-100. С early stopping: если loss на валидации перестает уменьшаться 5-10 эпох подряд, останавливаем обучение.

### Генерация (inference)

Авторегрессивная генерация: генерируем токен за токеном, каждый раз подавая предыдущий сгенерированный токен как вход.

1. Подаем стартовый токен `<BOS>`
2. LSTM выдает вероятности для первого слова
3. Выбираем слово (top-k sampling или temperature sampling)
4. Берем SVD-эмбеддинг выбранного слова
5. Подаем его на следующий шаг LSTM
6. Повторяем, пока не получим `<EOS>` или не достигнем максимальной длины

Стратегии выбора (top-k, temperature) -- те же, что описаны для варианта 1 (шаг 3.3).

#### 4.4. VAE -- опциональное расширение (Variational Autoencoder)

VAE -- не обязательный компонент, а опция. Можно реализовать основной вариант без VAE, а потом добавить VAE как улучшение, если останется время.

### Зачем нужен VAE

Обычный LSTM-генератор (из раздела 4.3) генерирует текст "с нуля": получает стартовый токен и дальше работает авторегрессивно. У него нет возможности "задать направление" генерации (кроме стартового скрытого состояния).

VAE добавляет латентное пространство -- пространство "смыслов". Каждый текст кодируется в маленький вектор z (32-64 числа). Разные z дают разные тексты. Можно:
- Сэмплировать случайный z и получить новый текст
- Взять z от одного текста и z от другого, найти точку посередине -- получить текст, который "смешивает" два исходных
- Целенаправленно двигаться в латентном пространстве в "юмористическом направлении"

Простая аналогия: VAE -- это как пульт управления настроением текста. Каждый ползунок на пульте (каждая компонента z) отвечает за какой-то аспект текста (тема, стиль, настроение). Двигая ползунки, мы получаем разные тексты.

### Архитектура VAE

```
Входная последовательность лемм
        |
        v
  [Encoder: LSTM]
  (читает последовательность SVD-эмбеддингов)
        |
        v
  Два выходных слоя:
    mu (среднее)         -- 32-64 чисел
    log_var (log дисперсии)  -- 32-64 чисел
        |
        v
  [Reparameterization trick]
    z = mu + exp(0.5 * log_var) * epsilon
    где epsilon ~ N(0, 1)
        |
        v
  [Decoder: LSTM]
  (принимает z как initial hidden state)
  (генерирует последовательность токенов)
        |
        v
  Восстановленная последовательность
```

### Reparameterization trick

Зачем: при обучении нейросети нужно вычислять градиенты. Если z -- случайная величина (сэмплированная из распределения), градиент через нее не пройдет. Трюк: вместо "сэмплировать z из N(mu, sigma^2)" пишем "z = mu + sigma * epsilon, где epsilon -- случайное число из N(0, 1)". Теперь epsilon -- фиксированный шум, а mu и sigma -- обучаемые параметры, через которые можно посчитать градиент.

Формула:
```
z = mu + exp(0.5 * log_var) * epsilon
```
Где `exp(0.5 * log_var) = sigma` (стандартное отклонение).

### Loss function VAE

VAE обучается минимизировать два слагаемых:

```
Loss = Reconstruction Loss + beta * KL Divergence
```

**Reconstruction Loss** -- cross-entropy между входной и восстановленной последовательностью. Штрафует за неточное восстановление текста.

**KL Divergence** -- мера того, насколько выученное распределение q(z|x) отличается от стандартного нормального N(0, 1). Штрафует за слишком "острые" распределения (когда модель кодирует каждый текст в точку, а не в облако). Формула:

```
KL = -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
```

**beta** -- коэффициент балансировки. При beta = 1 -- стандартный VAE. При beta < 1 -- больше акцент на реконструкцию (тексты точнее, но латентное пространство хуже организовано). При beta > 1 -- больше акцент на структуру латентного пространства (тексты менее точные, но генерация из случайных z дает более осмысленные результаты).

Рекомендация: начать с beta = 0.1 и постепенно увеличивать до 1.0 (beta annealing). Это помогает избежать posterior collapse.

### Что такое posterior collapse

Главная проблема VAE на малых данных. LSTM-декодер настолько мощный, что учится генерировать текст, полностью игнорируя z. Латентное пространство "коллапсирует": все z одинаковые (mu=0, sigma=1), и код z не несет информации о тексте.

Признак: KL Divergence падает до нуля, а Reconstruction Loss тоже уменьшается. Модель нашла "лазейку" -- научилась генерировать текст без z.

Как бороться:
- Beta annealing (постепенное увеличение beta от 0 до 1)
- Free bits (минимальный порог для KL, ниже которого штраф не начисляется)
- Маленький decoder (чтобы он не мог "все запомнить" без z)

Для 98 текстов риск posterior collapse высокий. Поэтому VAE -- опция, а не обязательный компонент.

#### 4.5. Оценка сложности варианта 2

### Плюсы

- **SVD-часть уже реализована.** В ноутбуке `Preprocessing_SVD_example.ipynb` есть функции `make_matrix_W_list_of_words()`, `apply_svd()`, `create_dictionary()`, `convert_text_to_vector()`. Нужна адаптация (TF-IDF -> BM25, русский -> македонский), но основной код готов.

- **Понятная математика.** SVD -- это линейная алгебра, ее проходят на 1-2 курсе. LSTM -- стандартная рекуррентная нейросеть. Нет экзотических компонент вроде GNN или attention в графе. Преподавателю и проверяющему будет легко понять, что происходит.

- **Не нужна torch_geometric.** Все зависимости стандартные: scipy, numpy, torch, faiss-cpu. Установка без сюрпризов.

- **Быстрое обучение.** SVD вычисляется за секунды (98 x 10 000 -- маленькая матрица). LSTM с frozen embedding layer обучается за минуты на CPU. Можно быстро перебирать гиперпараметры (k, hidden_size, num_layers).

- **Интерпретируемость SVD.** Сингулярные значения показывают, сколько "смысловых направлений" есть в корпусе. Столбцы V^T можно интерпретировать как "темы". Это хорошо ложится в курсовую работу: можно показать преподавателю, какие "темы" SVD нашла в македонских текстах.

- **Гибкая размерность.** SVD-эмбеддинги позволяют бесплатно экспериментировать с размерностью: вычислили SVD с k=97 один раз, а потом берем первые m компонент (m=20, 30, 50) без пересчета. У FastText такого нет -- там всегда 300 измерений.

### Минусы

- **Статичные эмбеддинги.** SVD-эмбеддинг слова не зависит от контекста. Слово "коса" (которое на македонском означает и волосы, и сельскохозяйственный инструмент) будет иметь один и тот же вектор во всех контекстах. Контекстные модели (BERT, GPT) решают эту проблему, но они основаны на Transformer, а Transformer на этапах обработки нам запрещен.

- **Не учитывает графовую структуру.** SVD работает с "мешком слов" (bag of words) -- порядок слов и связи между ними теряются. Семантический граф, который мы строили в подплане 3, здесь не используется напрямую. Графовые features можно добавить как дополнительные входы LSTM (конкатенировать к hidden state), но это будет слабее, чем GNN из варианта 1.

- **VAE -- риск posterior collapse.** Если добавлять VAE-расширение, на 98 текстах высока вероятность posterior collapse (LSTM-декодер проигнорирует латентный вектор z). Нужны специальные приемы (beta annealing, free bits), что усложняет код и отладку.

- **BM25-скоры "плоские".** BM25 учитывает только частоту слова в документе и его редкость в корпусе. Не учитывает порядок слов, синтаксис, семантические связи. SVD поверх BM25 может найти тематические кластеры, но не уловит тонкие синтаксические паттерны.

### Объем кода

Примерно 300-400 строк без VAE, ~500 строк с VAE:
- BM25-матрица и SVD-разложение (адаптация из ноутбука): ~100 строк
- Словарь эмбеддингов и FAISS-индекс: ~50 строк
- LSTM-генератор (модель + train loop + inference): ~200 строк
- VAE (encoder + decoder + reparameterization + loss): ~150 строк (опционально)

### Риск для курсовой: СРЕДНИЙ

Хороший баланс сложности и реализуемости. SVD-часть почти готова (адаптация ноутбука). LSTM -- стандартная архитектура с предсказуемым поведением. Основной риск -- качество генерации на 98 текстах может быть невысоким, но модель хотя бы будет работать (в отличие от варианта 1, где GNN может вообще не сойтись).

### Итоговая оценка варианта 2

Самый прагматичный вариант. Теория хорошо обоснована (книга LSM), код частично готов (ноутбук SVD), зависимости стандартные. Если нужен надежный результат в предсказуемые сроки -- это лучший выбор. VAE можно добавить как бонус, если останется время.

### Вариант 3: BM25 + нейросетевой генератор

#### 5.1. Общая схема варианта 3

Самый простой из трех вариантов. Идея: берем BM25-веса слов напрямую (без SVD-сжатия, без графов) и подаем их как дополнительный сигнал в LSTM-генератор. Embedding-и при этом обучаемые -- нейросеть сама учит представления слов.

Поток данных:

```
Корпус 98 текстов
       |
       v
  Лемматизация (CLASSLA)
       |
       v
  BM25-матрица (98 документов x ~10000 лемм)
       |
       v
  Для каждого слова в документе: BM25-вес
       |
       v
  +-----------------------+
  |  LSTM-генератор       |
  |                       |
  |  Embedding layer      |   <-- обучаемый (vocab_size x 256)
  |       |               |
  |       v               |
  |  weighted_embedding   |   <-- embedding * bm25_weight
  |       |               |
  |       v               |
  |  LSTM (1-2 слоя)      |   <-- hidden_size = 256
  |       |               |
  |       v               |
  |  Linear + softmax     |   <-- вероятность каждого слова
  +-----------------------+
       |
       v
  Сгенерированный текст на македонском
```

Ключевое отличие от варианта 2: нет промежуточного шага SVD. BM25-веса используются напрямую как множители для embedding-ов. Нет латентного пространства, нет эмбеддингов документов -- просто language model с дополнительным взвешиванием.

Ключевое отличие от варианта 1: нет графа и GNN. Нейросеть работает только с последовательностью слов и их BM25-весами, без учета структурных связей между концептами.

#### 5.2. Подготовка BM25-весов

### Что такое BM25-матрица

BM25-матрица -- это таблица размером (число документов) x (число лемм). Каждая ячейка содержит BM25-скор: числовую оценку важности слова для конкретного документа. Высокий скор означает, что слово характерно для этого документа (часто встречается здесь, но редко в других текстах).

Для нашего корпуса: 98 документов x примерно 10000 уникальных лемм = матрица 98 x 10000.

### Как используются BM25-веса при обучении

Для каждого слова в обучающем тексте:
1. Берем его embedding из обучаемого Embedding layer (вектор из 256 чисел)
2. Берем его BM25-скор из заранее посчитанной матрицы
3. Умножаем: `weighted_embedding = bm25_weight * embedding`

Зачем это нужно? BM25-вес играет роль "фильтра важности". Слова, характерные для текста (высокий BM25), получают усиленный embedding. Частые неинформативные слова (низкий BM25) получают ослабленный embedding. Нейросеть учится обращать внимание на содержательные слова.

### Реализация

BM25 можно посчитать двумя способами.

**Способ 1: pip-пакет `rank_bm25`**

```python
# установка: pip install rank-bm25
from rank_bm25 import BM25Okapi

# corpus_lemmas -- список документов, каждый документ -- список лемм
# например: [["мачка", "седи", "стол"], ["куче", "трча", "парк"], ...]
bm25 = BM25Okapi(corpus_lemmas)

# получаем BM25-скор для каждого слова в документе
# bm25.get_scores(query) -- считает BM25-скор запроса по всем документам
```

**Способ 2: своя реализация (~30 строк)**

```python
import math
from collections import Counter

def compute_bm25_matrix(corpus_lemmas, k1=1.5, b=0.75):
    # corpus_lemmas: список из N документов, каждый -- список лемм
    N = len(corpus_lemmas)
    # длины документов и средняя длина
    doc_lengths = [len(doc) for doc in corpus_lemmas]
    avgdl = sum(doc_lengths) / N
    # считаем document frequency для каждой леммы
    df = Counter()
    for doc in corpus_lemmas:
        for word in set(doc):
            df[word] += 1
    # собираем уникальный словарь
    vocab = sorted(df.keys())
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    # строим BM25-матрицу
    matrix = {}
    for doc_id, doc in enumerate(corpus_lemmas):
        tf = Counter(doc)
        dl = doc_lengths[doc_id]
        for word, freq in tf.items():
            idf = math.log((N - df[word] + 0.5) / (df[word] + 0.5) + 1)
            numerator = freq * (k1 + 1)
            denominator = freq + k1 * (1 - b + b * dl / avgdl)
            score = idf * numerator / denominator
            matrix[(doc_id, word)] = score
    return matrix, vocab, word_to_idx
```

### Хранение

BM25-скоры хранятся в двух местах:

1. **`nlp_data.db`** -- таблица `bm25_scores` с колонками `(text_id, lemma, score)`. Удобно для выборки по документу или по слову через SQL-запросы.

2. **`bm25_index.pkl`** -- сериализованный Python-объект (pickle). Быстрая загрузка в память при обучении нейросети. Формат: словарь `{(doc_id, lemma): score}` или numpy-матрица.

#### 5.3. Почему BM25, а не TF-IDF

### TF-IDF: формула и проблемы

```
TF-IDF(t, d) = tf(t, d) * log(N / df(t))
```

Где:
- `tf(t, d)` -- сколько раз слово t встретилось в документе d (сырая частота)
- `N` -- общее число документов
- `df(t)` -- в скольких документах встретилось слово t

Две проблемы:

**Проблема 1: нет насыщения частоты.** Если слово встретилось в документе 10 раз, его tf в 10 раз больше, чем если бы оно встретилось 1 раз. Но на практике 10-кратное повторение слова не означает, что оно в 10 раз важнее. После определенного порога повторения перестают добавлять смысл. TF-IDF этого не учитывает.

Частичное решение -- `sublinear_tf=True` в sklearn, тогда tf заменяется на `log(1 + tf)`. Но это грубое приближение.

**Проблема 2: нет нормализации по длине документа.** Длинный документ (роман на 50000 слов) и короткий рассказ (500 слов) обрабатываются одинаково. В длинном тексте частота слова будет выше просто потому, что текст длиннее, а не потому что слово важнее. TF-IDF этого не учитывает.

### BM25: формула и преимущества

```
BM25(t, d) = IDF(t) * (tf(t,d) * (k1 + 1)) / (tf(t,d) + k1 * (1 - b + b * dl/avgdl))
```

Где:
- `IDF(t) = log((N - df(t) + 0.5) / (df(t) + 0.5) + 1)` -- улучшенная версия IDF
- `k1` -- параметр насыщения (обычно 1.2-2.0, у нас 1.5)
- `b` -- параметр нормализации длины (обычно 0.75)
- `dl` -- длина документа d (в словах)
- `avgdl` -- средняя длина документа по всему корпусу

**Преимущество 1: насыщение частоты (параметр k1).** Формула `tf * (k1 + 1) / (tf + k1)` стремится к `(k1 + 1)` при больших tf. То есть вес слова растет с частотой, но постепенно выходит на плато. При k1 = 1.5 слово, встретившееся 5 раз, получит вес ~3.0, а встретившееся 50 раз -- всего ~2.5 (не в 10 раз больше, как при TF-IDF).

**Преимущество 2: нормализация по длине (параметр b).** Выражение `1 - b + b * dl/avgdl` масштабирует частоту с учетом длины текста. Если документ длиннее среднего (dl > avgdl), знаменатель увеличивается и вес снижается. Если короче -- наоборот. Параметр b = 0.75 означает, что длина учитывается сильно (но не полностью).

**Преимущество 3: лучше работает для коротких текстов.** Юмористические тексты часто короткие (анекдоты, короткие рассказы). В коротком тексте каждое слово встречается мало раз, и разница tf = 1 vs. tf = 2 существенна. BM25 с нормализацией длины корректно обрабатывает такие случаи, а TF-IDF нет.

### Связь BM25 и TF-IDF

TF-IDF -- это частный случай BM25. Если поставить k1 -> бесконечность (отключить насыщение) и b = 0 (отключить нормализацию длины), BM25 превращается в TF-IDF с IDF-компонентой.

Иными словами, BM25 -- это TF-IDF с двумя дополнительными "ручками настройки" (k1 и b), которые решают две конкретные проблемы.

### Почему BM25 выбран для нашего проекта

1. Корпус из 98 текстов очень разнородный по длине (короткие рассказы и длинные повести) -- нормализация длины (параметр b) критична
2. Юмористические элементы часто в коротких фрагментах -- насыщение частоты (параметр k1) не дает длинным текстам заглушить короткие
3. В книге LSM (глава 2) матрица W строится с нормализацией по длине и энтропийным весом -- BM25 делает то же самое, но более стандартным способом

#### 5.4. Нейросетевой генератор

### Embedding layer: обучаемый

В отличие от варианта 2 (где embedding-и фиксированы из SVD), здесь Embedding layer обучается вместе с остальной сетью. Это значит:

- Сеть начинает со случайных embedding-ов (случайные числа)
- В процессе обучения embedding-и постепенно настраиваются так, чтобы похожие слова оказались рядом в векторном пространстве
- К концу обучения сеть "выучила" свои собственные представления слов, оптимальные для задачи генерации

Параметры Embedding layer:
- `vocab_size` -- размер словаря (примерно 8000-12000 лемм после фильтрации по min_df и max_df)
- `embedding_dim` -- размерность embedding-а, обычно 128 или 256
- Общее число параметров: vocab_size * embedding_dim (например, 10000 * 256 = 2.56 млн параметров)

Плюс обучаемых embedding-ов: они оптимизируются именно под задачу генерации на македонском тексте. Минус: нужно достаточно данных для их обучения (у нас 98 текстов -- маловато).

### LSTM / GRU

Ядро генератора -- рекуррентная нейросеть. Два варианта:

**LSTM (Long Short-Term Memory)**:
- 1-2 слоя
- hidden_size = 256
- Число параметров на слой: 4 * (embedding_dim + hidden_size) * hidden_size = 4 * (256 + 256) * 256 = ~524 тыс.
- LSTM лучше запоминает длинные зависимости благодаря механизму "ворот" (gates): forget gate, input gate, output gate

**GRU (Gated Recurrent Unit)**:
- Упрощенная версия LSTM, 2 ворот вместо 3
- На 25% меньше параметров при том же hidden_size
- Часто работает не хуже LSTM, но быстрее обучается
- Можно выбрать GRU как запасной вариант, если LSTM переобучается

### Attention mechanism (опционально)

Attention позволяет генератору "смотреть назад" на предыдущие слова и выбирать, на какие из них обратить внимание при генерации следующего слова.

Без attention LSTM "видит" только свой hidden state -- сжатое резюме всех предыдущих слов. Чем длиннее текст, тем больше теряется информация о ранних словах.

С attention LSTM на каждом шаге вычисляет веса внимания к предыдущим словам и строит взвешенную сумму их скрытых состояний. Это простой attention (Bahdanau-style), а не multi-head self-attention из Transformer.

Важно: простой attention -- это не Transformer. Он не нарушает ограничение от преподавателя (запрет на Transformer на этапах обработки). Attention -- это просто взвешенная сумма, как BM25 -- просто взвешенная частота.

### Output: softmax

На выходе LSTM стоит линейный слой (Linear) размером hidden_size x vocab_size, за которым идет softmax. Softmax превращает выход линейного слоя в вероятности: для каждого слова из словаря -- число от 0 до 1, и сумма всех вероятностей равна 1.

При генерации выбираем следующее слово по этим вероятностям. Для юмора полезна повышенная температура (temperature = 1.2-1.5), которая делает распределение более "плоским" и увеличивает шанс выбрать менее вероятное, неожиданное слово.

### Архитектура в коде (PyTorch)

```python
class BM25LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim=256, hidden_size=256,
                 num_layers=2, dropout=0.3):
        super().__init__()
        # обучаемый Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # LSTM с dropout между слоями
        self.lstm = nn.LSTM(embedding_dim, hidden_size,
                            num_layers=num_layers,
                            dropout=dropout, batch_first=True)
        # линейный слой для предсказания следующего слова
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, bm25_weights, hidden=None):
        # x: (batch, seq_len) -- индексы слов
        # bm25_weights: (batch, seq_len) -- BM25-веса
        emb = self.embedding(x)  # (batch, seq_len, embedding_dim)
        # взвешиваем embedding-и BM25-весами
        bm25_weights = bm25_weights.unsqueeze(-1)  # (batch, seq_len, 1)
        weighted_emb = emb * bm25_weights  # поэлементное умножение
        # пропускаем через LSTM
        output, hidden = self.lstm(weighted_emb, hidden)
        # предсказываем следующее слово
        logits = self.fc(output)  # (batch, seq_len, vocab_size)
        return logits, hidden
```

Это ~30-40 строк основного кода. Остальное -- обучающий цикл, загрузка данных, генерация.

#### 5.5. Оценка сложности

### Плюсы

1. **Самый простой из трех вариантов.** Минимум движущих частей: BM25-матрица + обычный LSTM. Нет GNN, нет SVD, нет графов.

2. **Обучаемые embedding-и.** В отличие от варианта 2 (фиксированные SVD-embedding-и), здесь embedding-и учатся вместе с сетью. Это может дать лучшие представления слов для конкретной задачи генерации.

3. **BM25 считается дешево.** Подготовка BM25-матрицы -- несколько секунд на 98 текстах. Никаких тяжелых вычислений.

4. **Нет дополнительных зависимостей.** Только PyTorch (который нужен во всех трех вариантах). Не нужен torch-geometric (как в варианте 1), не нужен scipy для SVD (как в варианте 2, хотя scipy обычно уже стоит).

5. **Быстрое обучение.** На CPU: 20-40 минут для полного обучения.

### Минусы

1. **Не использует граф.** Семантический граф -- одна из ключевых частей проекта. Вариант 3 его полностью игнорирует. Графовые метрики (betweenness, incongruity, clustering), которые мы так тщательно считали, не используются.

2. **Не использует латентное пространство.** Нет SVD-сжатия, нет эмбеддингов документов в одном пространстве со словами. Теряется связь с книгой LSM.

3. **По сути -- обычный language model с BM25-взвешиванием.** Это стандартная архитектура, которую можно найти в любом учебнике по NLP. Для магистерской работы маловато оригинальности.

4. **98 текстов -- мало для обучения embedding-ов с нуля.** Обучаемые embedding-и требуют данных. На маленьком корпусе embedding-и могут не выучить полезных представлений.

5. **Менее интересен для магистратуры.** Теоретическая глубина невелика: BM25 -- стандартная формула из информационного поиска, LSTM -- стандартная архитектура. Нет связи ни с теорией графов (Barrat), ни с латентным семантическим анализом (LSM).

### Объём кода

Примерно 250-350 строк:
- BM25-подготовка: ~50 строк
- Модель (PyTorch): ~80 строк
- Обучающий цикл: ~60 строк
- Загрузка данных и токенизация: ~40 строк
- Генерация текста: ~30 строк

### Риск

Низкий. Все компоненты стандартные и хорошо протестированные. Главный риск -- не технический, а содержательный: модель может оказаться слишком простой для магистерской работы.

### Сравнительная таблица и выбор архитектуры

#### 6.1. Сравнительная таблица трех вариантов

| Критерий | Вариант 1: Graph+GNN | Вариант 2: BM25 -> SVD + LSTM | Вариант 3: BM25 + LSTM |
|---|---|---|---|
| Использует граф | Да (GNN encoder по вершинам и ребрам) | Нет (но графовые features можно добавить как доп. вход) | Нет |
| Использует LSA/SVD | Нет | Да (SVD-разложение BM25-матрицы) | Нет |
| Использует BM25 | Нет (embedding-и из FastText) | Да (BM25-матрица как вход для SVD) | Да (BM25-веса как множители embedding-ов) |
| Тип embedding-ов | Фиксированные (FastText 300d) + обогащение через GNN | Фиксированные (SVD, 50-97d) | Обучаемые (128-256d) |
| Encoder | GCN/GAT (2-3 слоя message passing) | SVD (линейная алгебра, не нейросеть) | Нет отдельного encoder-а |
| Decoder | LSTM с графовыми features на входе | LSTM с SVD-embedding-ами на входе | LSTM с BM25-взвешенными embedding-ами |
| Сложность реализации | Высокая | Средняя | Низкая |
| Объём кода (строки) | ~600-700 | ~350-450 | ~250-350 |
| Время обучения (CPU) | 2-4 часа | 30-60 минут | 20-40 минут |
| Риск переобучения | Высокий (много параметров, мало данных) | Средний (SVD -- фиксированные embedding-и, меньше параметров) | Средний (обучаемые embedding-и, но простая модель) |
| Теоретическая глубина | Высокая (теория графов + GNN) | Высокая (LSA/LSM + SVD) | Низкая (стандартный language model) |
| Связь с книгами | Barrat/Barthelemy (главы 1-3) | LSM Bellegarda (главы 1-5) | Частично LSM (только BM25 как замена TF-IDF) |
| Доп. зависимости | torch-geometric (PyG) | scipy, sklearn (обычно уже установлены) | Нет |
| Подходит для 98 текстов | С натяжкой (GNN нужны большие графы) | Да (SVD хорошо работает на маленьких корпусах) | Да (но 98 текстов мало для обучения embedding-ов) |
| Моделирование юмора | Через графовую структуру (incongruity между вершинами) | Через латентное пространство (далекие SVD-вектора) | Через температуру sampling-а (грубый подход) |

#### 6.2. Baseline-модели для сравнения

Чтобы понять, насколько хорошо работает выбранная нейросеть, нужны baseline-модели -- простые модели, с которыми мы будем сравнивать результаты. Если нейросеть не обогнала baseline, значит, сложная архитектура не оправдана.

### Baseline 1: Цепь Маркова (Markov chain)

**Суть:** для каждой пары слов (слово_A, слово_B) считаем, сколько раз слово_B идет после слова_A в корпусе. Получаем матрицу переходов. Генерация: выбираем стартовое слово, потом на каждом шаге случайно выбираем следующее слово по вероятностям переходов.

**Реализация:** ~50 строк кода.

```python
from collections import defaultdict, Counter
import random

# строим матрицу переходов из пар лемм
transitions = defaultdict(Counter)
for doc in corpus_lemmas:
    for i in range(len(doc) - 1):
        transitions[doc[i]][doc[i+1]] += 1

# генерация: выбираем следующее слово по вероятностям
def generate_markov(start_word, length=20):
    result = [start_word]
    current = start_word
    for _ in range(length):
        if current not in transitions:
            break
        candidates = transitions[current]
        total = sum(candidates.values())
        probs = {w: c/total for w, c in candidates.items()}
        next_word = random.choices(list(probs.keys()),
                                   weights=list(probs.values()))[0]
        result.append(next_word)
        current = next_word
    return ' '.join(result)
```

**Плюсы:**
- Простейшая модель, не требует обучения (только подсчет пар)
- Генерирует грамматически приемлемые пары слов
- Быстро работает

**Минусы:**
- Нет долгосрочной памяти: каждое слово зависит только от предыдущего
- Быстро скатывается в повторяющиеся циклы или бессвязный набор слов
- Не учитывает контекст дальше одного слова
- Не может выучить юмористические паттерны

### Baseline 2: N-gram language model

**Суть:** обобщение цепи Маркова. Вместо пар (bigram) используем тройки (trigram) или четверки (4-gram). Каждое следующее слово зависит не от одного предыдущего, а от N-1 предыдущих.

**Реализация:** ~100 строк кода.

```python
# bigram: P(w_i | w_{i-1})
# trigram: P(w_i | w_{i-2}, w_{i-1})
# сглаживание Лапласа: добавляем 1 к каждому счетчику,
# чтобы не было нулевых вероятностей
# P_smooth(w | context) = (count(context, w) + 1) / (count(context) + V)
# где V -- размер словаря
```

**Плюсы:**
- Чуть лучше цепи Маркова: учитывает контекст в 2-3 слова
- Сглаживание Лапласа решает проблему невиденных n-грамм
- Можно быстро посчитать perplexity для оценки

**Минусы:**
- На маленьком корпусе (98 текстов) trigram-модель будет очень разреженной: большинство троек встретятся 0 или 1 раз
- Нет обобщения: если тройка не встретилась в корпусе, модель ничего о ней не знает
- Не учитывает семантику слов (для модели "куче" и "мачка" -- просто два разных символа, без понимания, что оба -- животные)

### Зачем нужны baseline-модели

При оценке качества генерации мы будем считать одни и те же метрики (perplexity, BLEU, semantic coherence) для всех моделей: baseline-ов и нейросети. Если нейросеть генерирует текст с perplexity 150, а trigram-модель -- с perplexity 200, значит, нейросеть заметно лучше. Если разница минимальна -- значит, нейросеть не оправдывает свою сложность.

#### 6.3. Рекомендация по выбору архитектуры

**Рекомендация: Вариант 2 (BM25 -> SVD/LSA + LSTM) как основная модель.**

Ниже -- обоснование, почему вариант 2 рекомендуется, и что делать, если хочется другой вариант. Финальное решение принимает пользователь.

### Почему рекомендуется вариант 2

1. **Прямая связь с книгой LSM.** Вариант 2 буквально реализует то, что описано у Bellegarda в главах 1-5: строим матрицу W, применяем SVD, получаем embedding-и слов и документов в одном латентном пространстве. Это сильный теоретический фундамент для магистерской работы.

2. **SVD-код уже написан.** В `artifacts/Preprocessing_SVD_example.ipynb` уже есть рабочие функции: `make_matrix_W` (построение матрицы), `apply_svd` (SVD-разложение), `create_dictionary` (словарь embedding-ов), `convert_text_to_vector` (проекция текста в латентное пространство). Нужно адаптировать их под BM25 вместо TF-IDF и под наш корпус.

3. **Фиксированные embedding-и = меньше параметров.** SVD-embedding-и не обучаются -- они вычисляются один раз из матрицы W. У LSTM остается меньше параметров для обучения, и она меньше склонна к переобучению на 98 текстах.

4. **Средняя сложность.** Не слишком простой (как вариант 3), не слишком сложный (как вариант 1). Хороший баланс для магистерской.

### Рекомендуемая стратегия: вариант 2 + расширение

```
Этап 1: Вариант 2 (основная модель)
   -- BM25-матрица -> SVD -> LSTM генератор
   -- Срок: основная часть работы

Этап 2: Если останется время -- Graph Encoder (элементы варианта 1)
   -- Добавить графовые features как дополнительный вход LSTM
   -- Не полноценный GNN, а features вершин/ребер как доп. вектор
   -- Связь с книгой Barrat/Barthelemy

Запасной план: Если вариант 2 не заработает, откат на вариант 3
   -- Убрать SVD, оставить BM25 + LSTM
   -- Это подмножество варианта 2, откат простой
```

### Последствия выбора каждого варианта

**Если выбрать вариант 1 (Graph+GNN):**
- Плюс: самый теоретически глубокий, задействует всю графовую инфраструктуру
- Плюс: прямая связь с Barrat/Barthelemy
- Минус: самый сложный в реализации, высокий риск не успеть
- Минус: torch-geometric на CPU работает медленно
- Минус: 98 текстов -- мало для GNN (маленькие графы, мало примеров)
- Минус: если что-то пойдет не так, откат на более простой вариант займет время

**Если выбрать вариант 2 (SVD/LSA+LSTM):**
- Плюс: крепкая теоретическая база (LSM), код частично готов (SVD-ноутбук)
- Плюс: средний риск, средняя сложность
- Плюс: embedding-и фиксированы, LSTM учится быстро
- Плюс: легко расширить графовыми features (элементы варианта 1)
- Плюс: легко упростить до варианта 3 (откат)
- Минус: не использует графовую структуру напрямую (без расширения)

**Если выбрать вариант 3 (BM25+LSTM):**
- Плюс: самый простой, минимальный риск, быстро реализуется
- Плюс: обучаемые embedding-и могут выучить интересные представления
- Минус: теоретически слабый -- стандартный language model
- Минус: не использует ни граф, ни SVD -- зачем тогда строили графовый pipeline?
- Минус: для магистерской может выглядеть как "слишком просто"

### Важно

Это рекомендация, а не решение. Каждый вариант имеет свои плюсы и минусы. Выбор зависит от приоритетов:
- Если приоритет -- теоретическая глубина и графы -- вариант 1
- Если приоритет -- баланс между теорией и практикой -- вариант 2
- Если приоритет -- гарантированный результат с минимальным риском -- вариант 3

#### 6.4. Архитектура юмора

Независимо от выбранного варианта архитектуры, юмор в генерируемом тексте моделируется на нескольких уровнях.

### Уровень данных: обучение на юмористических элементах

Нейросеть учится из примеров. Если в обучающем корпусе есть тексты с юмористическими элементами, сеть подхватит их паттерны. Здесь важна разметка корпуса: нужно отметить, какие фрагменты текстов содержат юмор (ирония, гипербола, абсурд, каламбур).

На практике: при подготовке данных (plan_1) можно пометить предложения/абзацы тегами "humor" и "no_humor" и использовать эту разметку как дополнительный сигнал для нейросети.

### Уровень генерации: температура sampling-а

При генерации текста LSTM на каждом шаге выдает вероятности для каждого слова из словаря. Выбор слова зависит от температуры:

- **temperature = 1.0** -- стандартный sampling, вероятности как есть
- **temperature < 1.0** -- "холодный" sampling, модель чаще выбирает самое вероятное слово (предсказуемый, скучный текст)
- **temperature = 1.2-1.5** -- "горячий" sampling, распределение сглаживается, реже встречающиеся слова получают больше шансов

Для юмора полезна повышенная температура (1.2-1.5): она увеличивает вероятность неожиданных слов. Но слишком высокая температура (> 2.0) превращает текст в бессвязный набор слов.

Формула температурного sampling-а:

```
P(word_i) = exp(logit_i / T) / sum(exp(logit_j / T))
```

где T -- температура, logit_i -- выход линейного слоя перед softmax.

### Уровень графа: betweenness centrality и semantic incongruity

Этот уровень наиболее интересен и связан с теорией юмора (incongruity theory).

**Betweenness centrality** вершины показывает, насколько она -- "мост" между разными кластерами смыслов. Вершина с высоким betweenness соединяет семантически далекие группы концептов. В теории юмора это semantic incongruity -- столкновение несовместимых контекстов.

Как использовать при генерации:
1. Из обучающих графов выделить вершины с высоким betweenness centrality
2. При генерации давать таким вершинам (словам) повышенный вес
3. Нейросеть учится строить тексты, где "мостовые" слова связывают далекие концепты

**Semantic incongruity score** -- среднее косинусное расстояние между FastText-embedding-ами концов ребер. Чем выше, тем более неожиданные связи в графе.

Это можно использовать как целевую метрику при обучении: добавить в loss-функцию штраф за слишком низкий incongruity score, чтобы сеть стремилась генерировать тексты с неожиданными связями.

### Уровень метрик: FAISS-индекс для поиска далеких концептов

**FAISS** (Facebook AI Similarity Search) -- библиотека для быстрого поиска ближайших (и дальних) соседей в векторном пространстве.

Как это помогает юмору:
1. Загружаем FastText-embedding-и всех концептов в FAISS-индекс
2. Для выбранного концепта ищем не ближайших, а дальних соседей (по cosine distance)
3. Связь между далекими концептами -- потенциальный юмористический элемент

Пример: для концепта "мачка" (кошка) ближайшие соседи -- "куче" (собака), "животно" (животное). Далекие -- "весник" (газета), "компјутер" (компьютер), "политика" (политика). Связь "мачка" + "весник" = неожиданность = юмор.

### Уровень union edges: составные отношения

Модуль `higher_dim_graph.py` поддерживает union edges -- группировку нескольких ребер в одно составное отношение. Это мощный инструмент для моделирования юмора через incongruity theory.

**Принцип:** берем несколько нормальных ребер из разных семантических полей и объединяем их. Каждое ребро по отдельности нормально и ожидаемо. Но их совокупность создает абсурдную ситуацию.

**Пример на македонском:**

Ребро 1: "мачка" --седи на--> "стол" (кошка сидит на столе -- нормально)
Ребро 2: "мачка" --чита--> "весник" (кошка читает газету -- абсурд)

Каждое ребро по отдельности можно встретить в тексте. "Мачка седи на стол" -- обычная ситуация. "Чита весник" -- обычное действие (для человека). Но union edge, объединяющий оба ребра: "мачка седи на стол и чита весник" -- столкновение двух нормальных контекстов, которое создает абсурд. Это и есть incongruity theory в действии.

**Как использовать union edges:**
1. При анализе обучающих текстов: находить union edges, где компоненты принадлежат разным семантическим кластерам
2. Считать "incongruity score" union edge-а: среднее косинусное расстояние между embedding-ами агентов компонентных ребер
3. При генерации: комбинировать ребра из разных семантических полей, ориентируясь на высокий incongruity score

**Связь с графовыми метриками из Barrat:**
- Union edges -- это по сути гиперребра (hyperedges) в графе
- Они связывают более двух вершин одновременно
- Community detection по вершинам union edges может показать, какие семантические кластеры сталкиваются в юмористическом тексте

### Сводная таблица уровней юмора

| Уровень | Механизм | Что делает | Где реализуется |
|---|---|---|---|
| Данные | Разметка юмористических фрагментов | Нейросеть учится на примерах юмора | Подготовка данных (plan_1) |
| Генерация | Temperature sampling (1.2-1.5) | Повышает шанс неожиданных слов | Функция генерации в PyTorch |
| Граф: вершины | Betweenness centrality | "Мосты" между далекими кластерами | metrics_mk.py + features нейросети |
| Граф: ребра | Cosine distance embedding-ов | Семантическая неожиданность связей | semantic_incongruity_score() |
| Граф: union edges | Группировка ребер из разных полей | Абсурд через столкновение нормальных контекстов | higher_dim_graph.py |
| Метрики | FAISS-индекс далеких концептов | Поиск максимально неожиданных пар | faiss + FastText embedding-и |

## Шаг 7. Диаграмма потоков данных

Ниже — полная диаграмма, которая показывает путь от сырых текстов до сгенерированного юмористического текста. Каждый блок соответствует этапу обработки, стрелки — направление потока данных. Числа в скобках — реальные размерности из нашего корпуса (98 текстов после фильтрации).

```
┌──────────────────────────────────────────────────────────────────┐
│                      ВХОДНЫЕ ДАННЫЕ                             │
│  98 македонских художественных текстов (texts/)                 │
│  Отфильтрованы: только оригинальные мк-тексты (подплан 1)      │
└───────────────────────────┬──────────────────────────────────────┘
                            │
                            ▼
┌──────────────────────────────────────────────────────────────────┐
│            ЭТАП 1: ПРЕДОБРАБОТКА (подпланы 1-2)                 │
│  CLASSLA: tokenize → POS-tagging → lemma                       │
│  Результат: 5 485 683 токена, ~118 000 уникальных лемм         │
│  Хранение: таблица classla_tokens в nlp_data.db                │
└───────────────────────────┬──────────────────────────────────────┘
                            │
              ┌─────────────┴──────────────┐
              ▼                            ▼
┌───────────────────────┐    ┌─────────────────────────────────────┐
│  ВЕТКА A: ГРАФ        │    │  ВЕТКА B: ЭМБЕДДИНГИ                │
│  (подпланы 3, 5)      │    │  (этот подплан)                     │
│                       │    │                                     │
│  make_graph_mk.py:    │    │  1. Фильтрация лемм:               │
│  текст → семантич.    │    │     min_df=2, max_df=0.9            │
│  граф (вершины +      │    │              ↓                      │
│  рёбра)               │    │  2. BM25-матрица:                   │
│        ↓              │    │     98 docs × N лемм (sparse)       │
│  Метрики графа:       │    │              ↓                      │
│  density, centrality, │    │  3. SVD (k=50, диапазон 30-97):     │
│  clustering, degree   │    │     U(98×k), Σ(k), V^T(k×N)        │
│        ↓              │    │              ↓                      │
│  Графовые features    │    │  4. Словарь эмбеддингов:            │
│  (расширение, если    │    │     лемма → k-мерный вектор         │
│  останется время)     │    │     (вычисление: Σ · V^T)           │
│                       │    │              ↓                      │
│                       │    │  5. FAISS-индекс + nlp_data.db      │
└───────────┬───────────┘    └──────────────┬──────────────────────┘
            │                               │
            └──────────┬────────────────────┘
                       ▼
┌──────────────────────────────────────────────────────────────────┐
│            ЭТАП 2: LSTM-ГЕНЕРАТОР (этот подплан)                │
│                                                                  │
│  Embedding Layer: frozen SVD-эмбеддинги (vocab_size × k)        │
│        ↓                                                         │
│  LSTM (2 слоя, hidden_size=256, dropout=0.3)                    │
│        ↓                                                         │
│  Linear (256 → vocab_size) → Softmax                            │
│        ↓                                                         │
│  Sampling (temperature=1.0-1.5, top-k=50)                       │
└───────────────────────────┬──────────────────────────────────────┘
                            │
                            ▼
┌──────────────────────────────────────────────────────────────────┐
│            ЭТАП 3: ОЦЕНКА (подплан 7)                           │
│  Perplexity, TTR, MTLD, semantic incongruity                    │
│  Сравнение с baseline (Markov chain, n-gram)                    │
└───────────────────────────┬──────────────────────────────────────┘
                            │
                            ▼
┌──────────────────────────────────────────────────────────────────┐
│                         ВЫХОД                                    │
│  Сгенерированный юмористический текст на македонском            │
│  + визуализация графа (PyVis HTML)                              │
│  + таблица метрик качества                                      │
└──────────────────────────────────────────────────────────────────┘
```

### 7.1. Размерности данных на каждом этапе

Ниже — таблица с размерностями на каждом шаге pipeline. Буква N обозначает число лемм после фильтрации по min_df и max_df — точное значение определится при построении BM25-матрицы в шаге 8.

| Этап | Вход | Размерность входа | Выход | Размерность выхода |
|------|------|-------------------|-------|--------------------|
| CLASSLA | сырой текст (98 файлов) | ~5.5M символов | токены с леммами | 5 485 683 токена, 118 482 уникальных леммы |
| Фильтрация лемм | 118 482 леммы | min_df=2, без max_df | отфильтрованные леммы | N лемм (~46 700 без max_df) |
| BM25 | 98 текстов, N лемм | — | sparse-матрица W | 98 × N |
| SVD (k=50) | матрица W | 98 × N | U, Σ, V^T | U: 98×50, Σ: 50, V^T: 50×N |
| Эмбеддинги слов | V^T и Σ | 50×N и 50 | словарь: лемма → вектор | N векторов по 50 компонент |
| Sliding window (seq_len=64, шаг=32) | последовательности лемм | переменная длина | фрагменты | batch_size × 64 |
| Embedding Layer (frozen) | индексы слов | batch × seq_len | SVD-векторы | batch × seq_len × 50 |
| LSTM (2 слоя, hidden=256) | SVD-векторы | batch × seq_len × 50 | hidden states | batch × seq_len × 256 |
| Linear + Softmax | hidden states | batch × seq_len × 256 | вероятности | batch × seq_len × vocab_size |

### 7.2. Комментарий к архитектуре

Несколько ключевых решений, которые стоят за этой диаграммой:

**Frozen embedding layer.** SVD-эмбеддинги фиксируются (freeze=True) и не меняются при обучении LSTM. Причина: у нас всего 98 текстов — если сделать эмбеддинги обучаемыми, модель быстро переобучится на таком маленьком корпусе. Frozen-эмбеддинги выступают как "внешние знания" о семантике слов, полученные из статистики всего корпуса через SVD.

**LSTM вместо Transformer.** Ограничение от преподавателя: на этапах обработки текста (извлечение, токенизация, лемматизация, векторизация, индексация, поиск) запрещено использовать архитектуру Transformer. LSTM хорошо подходит для нашей задачи: корпус небольшой, последовательности не слишком длинные, а LSTM проще в реализации и отладке.

**BM25 вместо TF-IDF.** BM25 учитывает длину документа (параметр b) и имеет насыщение частоты (параметр k1). Слово, встретившееся 10 раз, не будет в 10 раз важнее, чем встретившееся 1 раз — это особенно важно для юмористических текстов разной длины. TF-IDF — частный случай BM25 (при k1→∞, b=0).

**Стартовое k=50.** Ранг SVD-разложения k определяет размерность эмбеддингов. У нас 98 документов, поэтому k ограничен сверху числом 97 (= 98 − 1). Стартовое значение k=50 — компромисс: достаточно для сохранения основных семантических измерений, но не слишком много, чтобы шум не "просочился" в эмбеддинги. Позже можно поэкспериментировать с k в диапазоне 30-97.

**Запасной план.** Если LSTM с frozen SVD-эмбеддингами не покажет хороших результатов (loss не падает, тексты бессмысленные), откатываемся на Вариант 3: BM25 + обучаемые эмбеддинги + LSTM. Там embedding layer обучается вместе с моделью, что может помочь на таком малом корпусе.

## Шаг 8. Прототип нейросети на PyTorch

### 8.1. Подготовка обучающих данных

На этом шаге мы превращаем сырые леммы из базы данных в готовые обучающие данные для LSTM-генератора. Работа идёт в несколько этапов:

1. Загружаем леммы из таблицы `classla_tokens` (результат подплана 2)
2. Фильтруем слишком редкие леммы (min_df=2 — слово должно встретиться хотя бы в 2 текстах)
3. Строим BM25-матрицу (98 документов x N отфильтрованных лемм)
4. Применяем SVD-разложение (k=50) и получаем k-мерные эмбеддинги для каждой леммы
5. Сохраняем эмбеддинги в FAISS-индекс (для быстрого поиска ближайших/дальних соседей) и в pickle
6. Строим vocabulary (словарь лемма → числовой индекс) со спец-токенами
7. Нарезаем тексты на перекрывающиеся фрагменты (sliding window)
8. Оборачиваем всё в PyTorch Dataset и DataLoader

**Почему без max_df.** В классическом information retrieval обычно фильтруют слишком частые слова (max_df=0.9) — они не помогают различать документы. Но мы строим языковую модель, которая генерирует текст. Союзы, предлоги и другие частые слова ("и", "на", "во", "е") составляют ~63% всех токенов. Без них модель не сможет строить грамматически правильные предложения. Поэтому мы оставляем все слова с min_df >= 2 и не ставим верхнюю границу.

После этого шага у нас будет всё необходимое для обучения нейросети.

In [ ]:
import sqlite3
import numpy as np
from collections import Counter, defaultdict
from rank_bm25 import BM25Okapi
from scipy.sparse.linalg import svds
from scipy.sparse import csr_matrix
import faiss
import pickle
import json as json_mod
import os
import time

# путь к базе данных (ноутбук лежит в source/, база — в datasets/)
db_path = '../datasets/nlp_data.db'
out_dir = '../datasets'

# подключаемся к базе с лемматизированными текстами
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# загружаем все леммы, сгруппированные по text_id
# фильтруем знаки препинания (PUNCT), числа (NUM) и символы (SYM) —
# они не несут смысла для языковой модели
cursor.execute(
    "SELECT text_id, lemma "
    "FROM classla_tokens "
    "WHERE upos NOT IN ('PUNCT', 'NUM', 'SYM') "
    "ORDER BY text_id, sentence_id, token_id"
)
rows = cursor.fetchall()

# собираем леммы каждого текста в отдельный список
docs_lemmas = defaultdict(list)
for text_id, lemma in rows:
    # приводим к нижнему регистру для единообразия
    docs_lemmas[text_id].append(lemma.lower())

# сортируем text_id для воспроизводимости
text_ids = sorted(docs_lemmas.keys())
# tokenized_corpus — список списков лемм, по одному списку на текст
tokenized_corpus = [docs_lemmas[tid] for tid in text_ids]

print(f"Загружено текстов: {len(text_ids)}")
print(f"Пример (текст {text_ids[0]}, первые 15 лемм): {tokenized_corpus[0][:15]}")

# считаем document frequency (DF) — в скольких текстах встречается каждая лемма
doc_freq = Counter()
for doc_lemmas in tokenized_corpus:
    # set() убирает дубликаты внутри одного текста — считаем только факт наличия
    for lemma in set(doc_lemmas):
        doc_freq[lemma] += 1

n_docs = len(text_ids)

# фильтруем леммы: оставляем слова, которые встречаются хотя бы в 2 текстах
# max_df НЕ используем — частые слова (союзы, предлоги) нужны языковой модели
# без них 63% токенов становятся <UNK>, и модель не может строить предложения
min_df = 2
filtered_vocab = sorted([
    lemma for lemma, freq in doc_freq.items()
    if freq >= min_df
])

print(f"\nФильтрация лемм:")
print(f"  min_df={min_df} (хотя бы в {min_df} текстах)")
print(f"  max_df — не применяем (частые слова нужны для генерации текста)")
print(f"  до фильтрации: {len(doc_freq)} уникальных лемм")
print(f"  после фильтрации: {len(filtered_vocab)} лемм")

# маппинг лемма -> позиция в отфильтрованном словаре
lemma2idx_bm25 = {lemma: i for i, lemma in enumerate(filtered_vocab)}

# создаем BM25-объект из всего корпуса
# BM25Okapi автоматически вычисляет IDF, среднюю длину документа и параметры k1, b
bm25 = BM25Okapi(tokenized_corpus)

# строим BM25-матрицу W[doc_i, word_j] = BM25 score слова j в документе i
# для каждого слова вычисляем его BM25-скор против всех 98 документов
n_words = len(filtered_vocab)
print(f"\nСтроим BM25-матрицу {n_docs} x {n_words}...")
t0 = time.time()

W = np.zeros((n_docs, n_words), dtype=np.float32)
for j, lemma in enumerate(filtered_vocab):
    if j % 10000 == 0 and j > 0:
        print(f"  обработано {j}/{n_words} лемм...")
    # get_scores([lemma]) возвращает массив из 98 скоров — по одному на документ
    scores = bm25.get_scores([lemma])
    W[:, j] = scores

elapsed = time.time() - t0
print(f"BM25-матрица готова за {elapsed:.1f} сек")
print(f"  shape: {W.shape}")
print(f"  ненулевых: {np.count_nonzero(W)} из {W.size} ({100*np.count_nonzero(W)/W.size:.1f}%)")

# SVD-разложение BM25-матрицы
# k=50 — стартовый ранг (ограничен сверху: k < min(98, n_words) = 97)
k = 50
print(f"\nSVD-разложение (k={k})...")

# преобразуем в sparse-матрицу для эффективности svds
W_sparse = csr_matrix(W)
u, sigma, vt = svds(W_sparse, k=k)

# svds возвращает сингулярные значения в произвольном порядке —
# сортируем по убыванию (от самого важного измерения к наименее важному)
desc_order = np.flip(np.argsort(sigma))
u = u[:, desc_order]       # U: (98, k) — эмбеддинги документов
sigma = sigma[desc_order]  # Sigma: (k,) — "важность" каждого измерения
vt = vt[desc_order]        # V^T: (k, n_words) — базис слов

print(f"  U: {u.shape} — эмбеддинги документов")
print(f"  Sigma: {sigma.shape} — сингулярные значения")
print(f"  V^T: {vt.shape} — базис слов")
print(f"  Топ-10 сингулярных значений: {sigma[:10].round(2)}")

# матрица эмбеддингов слов: (diag(Sigma) @ V^T).T
# каждая строка i — это k-мерный вектор для леммы filtered_vocab[i]
# умножение на Sigma взвешивает компоненты по их важности
word_embeddings = (np.diag(sigma) @ vt).T  # shape: (n_words, k)
print(f"\nМатрица эмбеддингов слов: {word_embeddings.shape}")
print(f"Примеры (первые 5 слов, первые 5 компонент):")
for i in range(min(5, n_words)):
    print(f"  {filtered_vocab[i]:20s} {word_embeddings[i, :5].round(4)}")

# нормализуем векторы для FAISS
# IndexFlatIP (Inner Product) на нормализованных векторах = cosine similarity
norms = np.linalg.norm(word_embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1  # защита от деления на ноль
word_embeddings_normed = (word_embeddings / norms).astype(np.float32)

# создаем FAISS-индекс для быстрого поиска ближайших соседей
index = faiss.IndexFlatIP(k)
index.add(word_embeddings_normed)
print(f"\nFAISS-индекс: {index.ntotal} векторов, размерность {k}")

# сохраняем FAISS-индекс на диск
faiss_path = os.path.join(out_dir, 'embeddings.faiss')
faiss.write_index(index, faiss_path)

# сохраняем маппинг lemma <-> faiss_id в таблицу embedding_index
cursor.execute("DROP TABLE IF EXISTS embedding_index")
cursor.execute(
    "CREATE TABLE embedding_index ("
    "lemma TEXT PRIMARY KEY, "
    "faiss_id INTEGER)"
)
for i, lemma in enumerate(filtered_vocab):
    cursor.execute("INSERT INTO embedding_index (lemma, faiss_id) VALUES (?, ?)", (lemma, i))
conn.commit()

# сохраняем pickle-копию словаря {лемма: вектор} (ненормализованный)
svd_dict = {lemma: word_embeddings[i] for i, lemma in enumerate(filtered_vocab)}
pkl_path = os.path.join(out_dir, 'svd_embeddings_mk.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(svd_dict, f)

# сохраняем документ-эмбеддинги (U) и сингулярные значения (Sigma)
np.save(os.path.join(out_dir, 'document_embeddings.npy'), u)
np.save(os.path.join(out_dir, 'svd_sigma.npy'), sigma)

# сохраняем BM25-матрицу и список лемм для воспроизводимости
np.save(os.path.join(out_dir, 'bm25_matrix.npy'), W)
with open(os.path.join(out_dir, 'filtered_vocab.json'), 'w', encoding='utf-8') as f:
    json_mod.dump(filtered_vocab, f, ensure_ascii=False)

print(f"\nСохранено:")
print(f"  {faiss_path} — FAISS-индекс ({index.ntotal} векторов)")
print(f"  {pkl_path} — pickle словарь")
print(f"  {out_dir}/document_embeddings.npy — U {u.shape}")
print(f"  {out_dir}/svd_sigma.npy — Sigma {sigma.shape}")
print(f"  {out_dir}/bm25_matrix.npy — BM25 матрица {W.shape}")
print(f"  {out_dir}/filtered_vocab.json — {len(filtered_vocab)} лемм")
print(f"  nlp_data.db:embedding_index — {len(filtered_vocab)} записей")

# тест: поиск ближайших соседей через FAISS
# берем слова со средней частотой для наглядной проверки
test_words = [w for w in ['селанец', 'копнеж', 'молитва', 'залак'] if w in lemma2idx_bm25]
if test_words:
    print(f"\nТест FAISS — 5 ближайших соседей:")
    for word in test_words:
        idx = lemma2idx_bm25[word]
        vec = word_embeddings_normed[idx:idx+1]
        # ищем 6 соседей (первый — само слово)
        distances, indices = index.search(vec, 6)
        neighbors = [
            (filtered_vocab[int(i)], f"{d:.3f}")
            for d, i in zip(distances[0], indices[0])
            if int(i) != idx
        ][:5]
        print(f"  {word} -> {neighbors}")

conn.close()


### 8.1.1. Что мы получили: BM25-матрица и SVD-эмбеддинги

**BM25-матрица** — это таблица размером 98 (текстов) на N (лемм, прошедших фильтрацию min_df=2), где каждая ячейка показывает, насколько важна конкретная лемма для конкретного текста. BM25 учитывает три вещи: как часто слово встречается в тексте (term frequency), насколько оно редкое в корпусе (inverse document frequency), и длину текста (короткий текст с тем же числом вхождений получает больший скор).

**SVD-разложение** сжимает эту большую матрицу в три маленьких: U (98 x 50) — "координаты" каждого текста в 50-мерном пространстве, Sigma (50) — "важность" каждого из 50 измерений, V^T (50 x N) — "координаты" каждого слова в тех же 50 измерениях. Умножая Sigma на V^T, мы получаем 50-мерный эмбеддинг для каждой леммы — компактное числовое представление смысла слова.

**FAISS-индекс** — библиотека от Meta для быстрого поиска похожих векторов. Мы нормализовали SVD-эмбеддинги и загрузили в IndexFlatIP (поиск по скалярному произведению = cosine similarity для нормализованных векторов). Теперь за доли секунды можно найти семантически близкие слова (для анализа) или максимально далёкие (для создания юмористических сочетаний — semantic incongruity).

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import json as json_mod
import numpy as np
import sqlite3
import pickle
import os

# пути к данным
db_path = '../datasets/nlp_data.db'
out_dir = '../datasets'

# загружаем отфильтрованный словарь лемм (из предыдущей ячейки или с диска)
vocab_path = os.path.join(out_dir, 'filtered_vocab.json')
with open(vocab_path, 'r', encoding='utf-8') as f:
    filtered_vocab = json_mod.load(f)

# загружаем SVD-эмбеддинги из pickle
pkl_path = os.path.join(out_dir, 'svd_embeddings_mk.pkl')
with open(pkl_path, 'rb') as f:
    svd_dict = pickle.load(f)

# размерность эмбеддингов (k из SVD)
embed_dim = next(iter(svd_dict.values())).shape[0]
print(f"SVD-эмбеддинги загружены: {len(svd_dict)} слов, размерность {embed_dim}")

# строим vocabulary: маппинг лемма <-> числовой индекс
# спец-токены занимают первые 4 позиции
PAD_IDX = 0   # padding — заполнитель для выравнивания длины последовательностей
BOS_IDX = 1   # beginning of sequence — начало текста
EOS_IDX = 2   # end of sequence — конец текста
UNK_IDX = 3   # unknown — неизвестное слово (нет SVD-эмбеддинга)

# word2idx: лемма -> числовой индекс
word2idx = {'<PAD>': PAD_IDX, '<BOS>': BOS_IDX, '<EOS>': EOS_IDX, '<UNK>': UNK_IDX}
# добавляем все леммы из отфильтрованного словаря
for lemma in filtered_vocab:
    word2idx[lemma] = len(word2idx)

# idx2word: числовой индекс -> лемма (обратный маппинг, нужен для декодирования)
idx2word = {idx: word for word, idx in word2idx.items()}

vocab_size = len(word2idx)
print(f"Vocabulary: {vocab_size} слов (включая 4 спец-токена)")

# сохраняем vocabulary на диск
vocab_save_path = os.path.join(out_dir, 'vocabulary_mk.json')
with open(vocab_save_path, 'w', encoding='utf-8') as f:
    json_mod.dump(word2idx, f, ensure_ascii=False)
print(f"Vocabulary сохранен: {vocab_save_path}")

# загружаем леммы из базы для построения обучающих последовательностей
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute(
    "SELECT text_id, lemma "
    "FROM classla_tokens "
    "WHERE upos NOT IN ('PUNCT', 'NUM', 'SYM') "
    "ORDER BY text_id, sentence_id, token_id"
)
rows = cursor.fetchall()
conn.close()

# группируем леммы по текстам
from collections import defaultdict
docs_lemmas = defaultdict(list)
for text_id, lemma in rows:
    docs_lemmas[text_id].append(lemma.lower())

text_ids = sorted(docs_lemmas.keys())

# конвертируем каждый текст в последовательность индексов
# слова без SVD-эмбеддинга заменяются на <UNK>
sequences_by_text = []
unk_count = 0
total_count = 0
for tid in text_ids:
    seq = []
    for lemma in docs_lemmas[tid]:
        total_count += 1
        if lemma in word2idx:
            seq.append(word2idx[lemma])
        else:
            seq.append(UNK_IDX)
            unk_count += 1
    sequences_by_text.append(seq)

print(f"\nКонвертация лемм в индексы:")
print(f"  всего токенов: {total_count}")
print(f"  <UNK> замен: {unk_count} ({100*unk_count/total_count:.1f}%)")

# sliding window: нарезаем каждый текст на перекрывающиеся фрагменты
# seq_len=64: длина фрагмента (сколько токенов подается модели за раз)
# step=32: шаг сдвига окна (перекрытие 50%)
# перекрытие нужно, чтобы каждый токен побывал в разных контекстах
seq_len = 64
step = 32

all_fragments = []  # список пар (input, target)

for seq in sequences_by_text:
    # добавляем <BOS> в начало и <EOS> в конец текста
    full_seq = [BOS_IDX] + seq + [EOS_IDX]

    # скользящее окно по всему тексту
    for start in range(0, len(full_seq) - seq_len, step):
        fragment = full_seq[start : start + seq_len + 1]  # +1, потому что target сдвинут
        # input = все токены кроме последнего
        inp = fragment[:-1]
        # target = все токены кроме первого (сдвиг на 1 вправо)
        tgt = fragment[1:]
        all_fragments.append((inp, tgt))

    # если текст короче seq_len — берем его целиком
    if len(full_seq) <= seq_len + 1:
        inp = full_seq[:-1]
        tgt = full_seq[1:]
        all_fragments.append((inp, tgt))

print(f"\nSliding window (seq_len={seq_len}, step={step}):")
print(f"  всего фрагментов: {len(all_fragments)}")

# train/val split: 90% train, 10% val
# фиксируем seed для воспроизводимости
np.random.seed(42)
indices = np.random.permutation(len(all_fragments))
split_idx = int(0.9 * len(indices))
train_fragments = [all_fragments[i] for i in indices[:split_idx]]
val_fragments = [all_fragments[i] for i in indices[split_idx:]]
print(f"  train: {len(train_fragments)}, val: {len(val_fragments)}")

# PyTorch Dataset — обертка, чтобы PyTorch мог перебирать данные
class MacedonianTextDataset(Dataset):
    # fragments — список пар (input, target), каждая пара — список индексов
    def __init__(self, fragments):
        self.fragments = fragments

    def __len__(self):
        # сколько примеров в датасете
        return len(self.fragments)

    def __getitem__(self, idx):
        # возвращаем один пример как пару тензоров
        inp, tgt = self.fragments[idx]
        return torch.tensor(inp, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

# collate_fn — собирает батч из отдельных примеров
# фрагменты могут быть разной длины (короткие тексты),
# поэтому дополняем короткие нулями (<PAD>) до максимальной длины в батче
def collate_fn(batch):
    inputs, targets = zip(*batch)
    # находим максимальную длину в этом батче
    max_len = max(inp.size(0) for inp in inputs)
    # создаем тензоры, заполненные нулями (PAD_IDX=0)
    padded_inputs = torch.zeros(len(inputs), max_len, dtype=torch.long)
    padded_targets = torch.zeros(len(targets), max_len, dtype=torch.long)
    # копируем реальные данные в начало каждой строки
    for i, (inp, tgt) in enumerate(zip(inputs, targets)):
        padded_inputs[i, :inp.size(0)] = inp
        padded_targets[i, :tgt.size(0)] = tgt
    return padded_inputs, padded_targets

# создаем Dataset и DataLoader
batch_size = 32
train_dataset = MacedonianTextDataset(train_fragments)
val_dataset = MacedonianTextDataset(val_fragments)

# shuffle=True для train — перемешиваем каждую эпоху для лучшего обучения
# shuffle=False для val — валидация должна быть воспроизводимой
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"\nDataLoader:")
print(f"  batch_size: {batch_size}")
print(f"  train батчей: {len(train_loader)}")
print(f"  val батчей: {len(val_loader)}")

# строим embedding matrix — матрицу весов для nn.Embedding
# каждая строка i — SVD-эмбеддинг для слова с индексом i
embedding_matrix = np.zeros((vocab_size, embed_dim), dtype=np.float32)

# спец-токены: <PAD> — нулевой вектор (не несет информации),
# <BOS>, <EOS>, <UNK> — маленькие случайные векторы
np.random.seed(42)
embedding_matrix[BOS_IDX] = np.random.normal(0, 0.01, embed_dim)
embedding_matrix[EOS_IDX] = np.random.normal(0, 0.01, embed_dim)
embedding_matrix[UNK_IDX] = np.random.normal(0, 0.01, embed_dim)

# заполняем SVD-эмбеддингами для всех слов из словаря
filled_count = 0
for word, idx in word2idx.items():
    if word in svd_dict:
        embedding_matrix[idx] = svd_dict[word]
        filled_count += 1

print(f"\nEmbedding matrix: {embedding_matrix.shape}")
print(f"  заполнено SVD-эмбеддингами: {filled_count} из {vocab_size}")

# оборачиваем в nn.Embedding с freeze=True —
# веса не будут меняться при обучении (frozen embedding layer)
# padding_idx=0 — вектор для <PAD> всегда остается нулевым
embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(embedding_matrix),
    freeze=True,
    padding_idx=PAD_IDX
)
print(f"  nn.Embedding: freeze=True, padding_idx={PAD_IDX}")

# проверяем один батч
sample_input, sample_target = next(iter(train_loader))
print(f"\nПример батча:")
print(f"  input shape: {sample_input.shape}")   # (batch_size, seq_len)
print(f"  target shape: {sample_target.shape}")  # (batch_size, seq_len)

# прогоняем через embedding layer
sample_embedded = embedding_layer(sample_input)
print(f"  embedded shape: {sample_embedded.shape}")  # (batch_size, seq_len, embed_dim)

# декодируем первые 15 токенов первого примера обратно в леммы
first_seq = sample_input[0].tolist()
decoded = [idx2word.get(idx, '?') for idx in first_seq[:15]]
print(f"\nПервые 15 токенов первого примера:")
print(f"  индексы: {first_seq[:15]}")
print(f"  леммы:   {decoded}")

# сохраняем embedding matrix для будущего использования
emb_matrix_path = os.path.join(out_dir, 'embedding_matrix.npy')
np.save(emb_matrix_path, embedding_matrix)
print(f"\nEmbedding matrix сохранена: {emb_matrix_path}")

# итоговая сводка
print(f"\n{'='*60}")
print(f"Подготовка данных завершена:")
print(f"  vocab_size = {vocab_size}")
print(f"  embed_dim = {embed_dim} (SVD k)")
print(f"  seq_len = {seq_len}")
print(f"  train примеров: {len(train_fragments)}")
print(f"  val примеров: {len(val_fragments)}")
print(f"  train батчей: {len(train_loader)}")
print(f"  val батчей: {len(val_loader)}")
print(f"  UNK токенов: {unk_count} ({100*unk_count/total_count:.1f}%)")
print(f"  embedding: frozen SVD-эмбеддинги")
print(f"{'='*60}")


### 8.1.2. Что такое sliding window, Dataset и embedding matrix

**Sliding window (скользящее окно)** — приём для увеличения числа обучающих примеров. Мы берём текст, ставим "окно" длиной 64 токена в начало, вырезаем фрагмент, затем сдвигаем окно на 32 токена вперёд и вырезаем следующий. Перекрытие в 50% нужно, чтобы каждый токен побывал в разных позициях: в начале фрагмента, в середине и в конце. Из каждого фрагмента делаем пару: input (все токены кроме последнего) и target (все токены кроме первого) — модель учится предсказывать следующий токен по предыдущим.

**Dataset и DataLoader** — стандартные инструменты PyTorch для работы с данными. Dataset хранит все примеры и умеет отдавать по одному. DataLoader группирует примеры в батчи (по 32 штуки), перемешивает их перед каждой эпохой и подготавливает для подачи в нейросеть. Функция collate_fn дополняет короткие фрагменты нулями (padding) до длины самого длинного в батче — так все примеры в батче имеют одинаковую длину.

**Embedding matrix (frozen)** — матрица размером vocab_size x 50, где каждая строка — SVD-эмбеддинг слова. Параметр freeze=True означает, что эти веса не меняются при обучении: модель использует их "как есть", без подстройки. Это важно, потому что у нас всего 98 текстов — если разрешить эмбеддингам обучаться, они переобучатся (запомнят обучающие данные вместо общих закономерностей). Frozen-эмбеддинги работают как "внешние знания" о семантике слов, извлечённые из всего корпуса через BM25 + SVD.